# Archived analysis workflow

This notebook records the author workflow used during revision. It requires the corresponding external model-ready inputs, saved predictions and/or checkpoints. Its `/content/drive/MyDrive` paths describe the author environment and must be reviewed before execution. Do not run all cells merely to inspect the repository. See [analysis/README.md](../README.md) and the main repository README for the supported benchmark entry point.

This publication-scoped copy omits unused component-test switches in its embedded helpers. The full-model k-size computation is retained. Its workflow fingerprint is new: use a new output directory, not an existing author run directory.


# SpaGAT：邻域大小敏感性 + 距离单位核对（v2）

**本 notebook 新增4次训练。** SEA-AD、Mouse 分别训练 k=25、75，固定 seed=123；复用 Figure 2 中同一 seed 的 k=50 完整模型。此处 k 是含中心细胞的总细胞数，对应24、49、74个周围邻居，与 routing 的4个通道不同。

- 保持主实验固定的 train/validation/test、训练集 baseline、细胞/基因顺序、完整架构、loss、优化器与训练预算。每个 k 最多10 epochs，patience=5，batch=32；validation PCC选checkpoint，之后评估test。
- 三个 k 预先固定，全部报告；不根据 test挑选k，也不默认k=50最优。只有 seed=123，不能给出训练seed SD。
- k=25取原邻域前25个；k=75保留原50个，再补同一切片内最近25个。先验证原邻域、索引和目标一致性。只保存新的邻居索引，原CSV、baseline、split和模型均不覆盖。
- 按之前消融速度粗估4次训练约3–5小时，**另加首次输入核对、邻域构建、Drive读写和推理**；这不是已实测时长。GPU型号、IO和early stopping均会影响时间。
- 单位审计读取MERFISH/CosMx相关记录与代码片段。模型距离阶段不额外缩放坐标。**坐标值范围或脚本注释不能单独证明μm单位**，摘要会将缺少上游转换证据的项目列为待核实。
- 当前仅核对Liver坐标记录，不新增Liver训练。随机基因/routing对照是另一个实验，本notebook不会运行它们。

新结果写入时间戳Drive目录。完成后返回小型summary.zip；checkpoint、预测矩阵和邻居索引留在Drive。不要先替换论文结果。

本地CPU合成测试已通过：三个k的训练→validation选模→test导出、checkpoint预测复现、邻居距离核对、原50邻居保留、目标/划分不变与同参数量；k=50前向输出逐值一致。尚未在真实Colab/GPU数据上运行。

**v2检查修正：** 原预处理对x、y方向距离差做了10,000截断。v1检查所有行的未经截断距离，可能因此在原本已排除的孤立receiver行上停下。v2使用原flag和loader索引确定实际receiver，并逐切片记录行号、原始距离倒序幅度、是否触发截断及旧距离规则检查结果。有效receiver仍需通过原来的严格检查，不放宽容差；已排除的receiver不进入训练/评估，作为其他细胞的sender资格保持原样。不根据模型表现改变邻域。

**首次运行v2：** RESUME_DIR保持None，产生新目录；不要填写v1失败目录。该报错发生在训练前，未损失新增训练结果。当前尚未拿到真实报错行，v2通过与否将确认异常是否只涉及已排除receiver；若有效receiver异常，仍停止并输出详细诊断。

以后v2断线：填写v2打印的目录到RESUME_DIR，从第1格重新运行。完成任务会跳过；中断任务保留日志，并只重跑该任务。不要填写旧消融目录或主实验目录。


## 1. 选择GPU并挂载Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch,numpy,pandas,scipy
if not torch.cuda.is_available():
    raise RuntimeError('请在 Runtime → Change runtime type 中选择 GPU。')
print('GPU:',torch.cuda.get_device_name())


## 2. 输出目录与断线续跑

In [ ]:
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/SpaGAT_revision/inductive_training/20260922_104353_552891_UTC_f291de00')
RESUME_DIR=None  # 首次保持None；断线后填写本次neighborhood_sensitivity_...目录


## 3. 加载固定代码与配置

In [ ]:
REFERENCES=[{'dataset': 'Mouse', 'model': 'SpaGAT', 'seed': 123, 'npz': '/content/drive/MyDrive/SpaGAT_revision/inductive_training/20260922_104353_552891_UTC_f291de00/results/Mouse/seed123_test/test_predictions.npz', 'bytes': 176293041, 'target_sha256_float32': '020e53bc1d45232be8eac049babd96e4b8bd9a33e5a40bd198ef309e8fb1dd38', 'prediction_sha256_float32': '987f81a334998b12fc55ccd991c55aed91649e104436484a83e8f2fa374cb1be', 'metrics_sha256': 'b448611ac98abce73830caffbe5eab903b26db004695b4c73c47ce9354a377c8', 'split_path': '/content/drive/MyDrive/SpaGAT_revision/inductive_training/20260922_104353_552891_UTC_f291de00/splits/Mouse_inductive_split_indices.json', 'split_sha256': 'd7ec406fc3be3cf2aba12c53d0e9a9d901ed830f9827554e901fed494f8bb113', 'samples_path': '/content/drive/MyDrive/SpaGAT_revision/inductive_training/20260922_104353_552891_UTC_f291de00/splits/Mouse_inductive_split_indices_samples.csv', 'samples_sha256': 'f3d3022be6bd09f519834e2b7ea0cc614542e59ea3a0fe98de54f4332f3c5e4e', 'checkpoint_path': '/content/drive/MyDrive/SpaGAT_revision/inductive_training/20260922_104353_552891_UTC_f291de00/results/Mouse/spagat_seed123_best.pth'}, {'dataset': 'SEA_AD', 'model': 'SpaGAT', 'seed': 123, 'npz': '/content/drive/MyDrive/SpaGAT_revision/inductive_training/20260922_104353_552891_UTC_f291de00/results/SEA_AD/seed123_test/test_predictions.npz', 'bytes': 43033345, 'target_sha256_float32': '58613a068b692e8621b8c3efaa93939c34b83abdd7aa060a74f0d0d68de7083e', 'prediction_sha256_float32': '0f01da16dd33ff71822a1899a58cb656437e4ed9b55725b8d8cd4bc12c13dc77', 'metrics_sha256': '86bcbae03e99fb9f0478ef282f16594d02481dc5550cd27513e59ed3d2373f10', 'split_path': '/content/drive/MyDrive/SpaGAT_revision/inductive_training/20260922_104353_552891_UTC_f291de00/splits/SEA_AD_inductive_split_indices.json', 'split_sha256': 'b3536c1b8ca6b198a588ae1fa2b846b06214106c9f687200c79511f3d5b8250c', 'samples_path': '/content/drive/MyDrive/SpaGAT_revision/inductive_training/20260922_104353_552891_UTC_f291de00/splits/SEA_AD_inductive_split_indices_samples.csv', 'samples_sha256': '56f6a99b6b71149b7b17116198c34149d5f34e19e71ae811fe28a9b807c2be54', 'checkpoint_path': '/content/drive/MyDrive/SpaGAT_revision/inductive_training/20260922_104353_552891_UTC_f291de00/results/SEA_AD/spagat_seed123_best.pth'}]
PROTOCOL={'datasets': ['Mouse', 'SEA_AD'], 'seed': 123, 'k_values': [25, 50, 75], 'k_includes_receiver': True, 'reuse_k50_seed123': True, 'new_training_runs': 4, 'max_epochs': 10, 'patience': 5, 'batch_size': 32, 'lr': 0.0001, 'neighborhoods': 'Keep ordered original first 50; truncate to 25 or append 25 nearest remaining cells in the same section. Verify exact Euclidean nearest-neighbor ordering first.', 'same_cells_targets_genes_splits_baselines': True, 'coordinate_scaling_changed': False, 'test_used_to_select_k': False, 'ligand_and_routing_controls_not_included': True, 'preflight_revision': 'v2_eligible_receiver_order_and_legacy_clipping_diagnostics', 'eligibility_scope': 'Use the original saved flags and frozen loader receiver indices. Excluded focal-receiver rows remain stored; distance-order findings are logged and do not change any included graph.', 'scope': 'Single-seed sensitivity; no seed SD; no claim of optimal k or identical physical radii across platforms.'}
SOURCES={'ablation_support.py': '"""Audit and summarize a fixed, predeclared component comparison."""\nfrom pathlib import Path\nimport hashlib, json, importlib.util, sys\nimport numpy as np\nimport pandas as pd\n\nVARIANTS = {\'no_distance\': {\'n_programs\': 4, \'routing_mode\': \'program\', \'definition\': \'All five distance input features are fixed to zero throughout training and evaluation; the original 50-cell neighborhoods, ordering, ligand inputs and all layers are retained. This tests explicit distance features conditional on the spatial graph.\'}}\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'): h.update(b)\n    return h.hexdigest()\n\ndef array_sha(a):\n    # Match the existing Figure 2/R3.3 provenance format exactly.\n    a=np.ascontiguousarray(a,dtype=np.float32)\n    h=hashlib.sha256(str(a.shape).encode())\n    h.update(a.tobytes())\n    return h.hexdigest()\n\ndef save_json(p,x):\n    Path(p).write_text(json.dumps(x,indent=2,ensure_ascii=False,allow_nan=False))\n\ndef check_reference_files(records):\n    """Check explicit training-record paths; never infer weights from NPZ location."""\n    missing=[]\n    for rec in records:\n        label=f"{rec[\'dataset\']} seed={rec[\'seed\']}"\n        if not rec.get(\'checkpoint_path\'):\n            raise ValueError(f\'{label}: explicit checkpoint_path is required; reload the v3 notebook.\')\n        ckpt=Path(rec[\'checkpoint_path\'])\n        if ckpt.name!=f"spagat_seed{rec[\'seed\']}_best.pth":\n            raise ValueError(f\'{label}: checkpoint filename does not match seed: {ckpt}\')\n        for field in [\'checkpoint_path\',\'npz\',\'split_path\',\'samples_path\']:\n            p=Path(rec[field])\n            if not p.is_file():missing.append(f\'{label} {field}: {p}\')\n    if missing:\n        raise FileNotFoundError(\'Missing reference files; no training started and no substitute selected:\\n\'+\'\\n\'.join(missing))\n    return [{\'dataset\':r[\'dataset\'],\'seed\':r[\'seed\'],\n             \'checkpoint_path\':r[\'checkpoint_path\'],\'prediction_path\':r[\'npz\']} for r in records]\n\ndef metrics(p,y):\n    p=np.asarray(p,dtype=np.float64);y=np.asarray(y,dtype=np.float64)\n    if p.shape!=y.shape or p.ndim!=2 or not np.isfinite(p).all() or not np.isfinite(y).all():\n        raise ValueError(\'Invalid prediction/target arrays\')\n    pc=p-p.mean(0);yc=y-y.mean(0)\n    denom=np.sqrt(np.maximum(np.sum(pc*pc,0)*np.sum(yc*yc,0),1e-12))\n    corr=np.clip(np.sum(pc*yc,0)/denom,-1,1)\n    mse=float(np.mean((p-y)**2));mse0=float(np.mean(y*y))\n    if mse0<=0:raise ValueError(\'Zero MSE0\')\n    return {\'median_gene_pcc\':float(np.sort(corr)[(len(corr)-1)//2]),\n            \'mse\':mse,\'mse0\':mse0,\'ev_zero_percent\':100*(1-mse/mse0)},corr\n\ndef load_reference(rec,split):\n    with np.load(rec[\'npz\'],allow_pickle=False) as z:\n        p=z[\'prediction\'];y=z[\'target\'];genes=z[\'genes\'].astype(str)\n        if not np.array_equal(z[\'test_indices\'],split[\'test_indices\']):raise ValueError(\'Reference test order changed\')\n        if array_sha(y)!=rec[\'target_sha256_float32\']:raise ValueError(\'Reference target hash changed\')\n        if array_sha(p)!=rec[\'prediction_sha256_float32\']:raise ValueError(\'Reference prediction hash changed\')\n        return p,y,genes\n\ndef import_train(source):\n    sys.path.insert(0,str(source))\n    spec=importlib.util.spec_from_file_location(\'fixed_train\',Path(source)/\'scripts/train.py\')\n    m=importlib.util.module_from_spec(spec);spec.loader.exec_module(m)\n    return m\n\ndef build_model(genes,ligands,variant=\'full\'):\n    from spagat.gene_program_model import SpaGP\n    v=VARIANTS.get(variant,{\'n_programs\':4,\'routing_mode\':\'program\'})\n    m=SpaGP(genes,ligands,node_dim=256,edge_dim=48,num_heads=2,n_layers=1,att_dim=8,\n            n_programs=v[\'n_programs\'],program_aware=True,program_token_dim=32,\n            message_decoder=\'free\',routing_mode=v[\'routing_mode\'])\n    m.embeddings.component_ablation=variant\n    return m\n\ndef synthetic_checks():\n    import torch\n    from spagat.gene_program_model import SpaGP_Loss\n    torch.manual_seed(23)\n    genes=[\'g\'+str(i) for i in range(8)];ligands=([[\'g0\'],[\'g1\']],[[0],[0]])\n    x={\'x\':torch.rand(2,50,8)*.1,\'type_exp\':torch.ones(2,50,8),\n       \'cell_types\':torch.randint(0,3,(2,50)), \'position_x\':torch.rand(2,50)*40,\n       \'position_y\':torch.rand(2,50)*40,\'y\':torch.rand(2,8),\'neighbor_mask\':torch.ones(2,50)}\n    report={}\n    for variant in [\'full\',*VARIANTS]:\n        model=build_model(genes,ligands,variant)\n        seen=[]\n        hook=model.embeddings.ligands_encoder.register_forward_pre_hook(lambda m,a:seen.append(a[0].detach().clone()))\n        result=model(x);hook.remove()\n        loss=SpaGP_Loss(genes,ligands,lambda_orth=.1,lambda_sparse=.01)(result,x[\'y\'])[0]\n        if not torch.isfinite(loss):raise ValueError(\'Nonfinite synthetic loss \'+variant)\n        loss.backward()\n        if any(p.grad is not None and not torch.isfinite(p.grad).all() for p in model.parameters()):raise ValueError(\'Nonfinite gradient\')\n        if variant==\'no_distance\':\n            altered=dict(x,position_x=x[\'position_x\']*37+10,position_y=-x[\'position_y\']*13)\n            with torch.no_grad():assert torch.equal(model(x)[0],model(altered)[0]),\'Distance dependence remains\'\n        report[variant]={\'parameters\':sum(p.numel() for p in model.parameters()),\'finite_loss_and_gradients\':True}\n    assert report[\'full\'][\'parameters\']==report[\'no_distance\'][\'parameters\']\n    return report\n\ndef full_preflight(config):\n    import torch\n    from torch.utils.data import DataLoader,Subset\n    source=Path(config[\'source\']);train=import_train(source)\n    from spagat.dataloader import SPAGAT_dataset\n    ds=config[\'dataset\'];records=config[\'references\']\n    check_reference_files(records)\n    split_path=Path(records[0][\'split_path\']);split=json.loads(split_path.read_text())\n    data=train.SPAGAT_dataset(split[\'processed_dir\'],num_neighbors=50)\n    train.load_and_validate_shared_split(split_path,len(data),split[\'processed_dir\'])\n    train.validate_sample_order(str(split_path),data,split)\n    # Collect actual test targets directly from the frozen loader\'s tensors.\n    # This avoids unnecessary graph construction, and checks every test target.\n    ix=np.asarray(split[\'test_indices\'],dtype=np.int64)\n    ends=np.cumsum(data.meta_counts);starts=ends-np.asarray(data.meta_counts)\n    sample_id=np.searchsorted(ends,ix,side=\'right\')\n    target=np.empty((len(ix),len(data.genes)),dtype=np.float32)\n    for j in range(len(data.samples)):\n        which=np.where(sample_id==j)[0]\n        if len(which):\n            row=data.arg_meta[j][torch.as_tensor(ix[which]-starts[j],dtype=torch.long)]\n            target[which]=data.exps[j][row].numpy()\n    if array_sha(target)!=records[0][\'target_sha256_float32\']:raise ValueError(\'Current loader targets differ from Figure 2\')\n    report={\'dataset\':ds,\'all_test_targets_match\':True,\'full_runs\':[],\'synthetic_checks\':synthetic_checks()}\n    device=torch.device(\'cuda\')\n    batch=next(iter(DataLoader(Subset(data,split[\'test_indices\'][:32]),batch_size=32,shuffle=False)))\n    batch={k:v.to(device) for k,v in batch.items()}\n    for rec in records:\n        p,y,genes=load_reference(rec,split)\n        if list(genes)!=list(map(str,data.genes)):raise ValueError(\'Gene order mismatch\')\n        ckpt=Path(rec[\'checkpoint_path\'])\n        saved=torch.load(ckpt,map_location=\'cpu\',weights_only=False)\n        expected={\'max_epochs\':10,\'patience\':5,\'batch_size\':32,\'lr\':1e-4,\'num_neighbors\':50,\n                  \'node_dim\':256,\'edge_dim\':48,\'num_heads\':2,\'n_layers\':1,\'att_dim\':8,\n                  \'n_programs\':4,\'program_token_dim\':32,\'lambda_orth\':.1,\'lambda_sparse\':.01}\n        for k,v in expected.items():\n            if saved[\'config\'].get(k)!=v:raise ValueError(f\'Full config mismatch {ds}/{rec["seed"]}/{k}\')\n        if saved[\'seed\']!=rec[\'seed\']:raise ValueError(\'Checkpoint seed differs\')\n        oldsplit=saved[\'shared_split\']\n        for key in [\'train_indices\',\'val_indices\',\'test_indices\']:\n            if oldsplit[key]!=split[key]:raise ValueError(\'Checkpoint split differs\')\n        model=build_model(data.genes,data.interactions).to(device)\n        model.load_state_dict(saved[\'model\'],strict=True);model.eval()\n        with torch.no_grad():test=model(batch)[0].cpu().numpy()\n        if not np.allclose(test,p[:len(test)],rtol=1e-4,atol=1e-4):raise ValueError(\'Frozen full model does not reproduce saved prediction batch\')\n        m,_=metrics(p,y)\n        report[\'full_runs\'].append({\'seed\':rec[\'seed\'],\'checkpoint_path\':str(ckpt),\n          \'reference_prediction_path\':rec[\'npz\'],\'checkpoint_sha256\':sha(ckpt),\n          \'prediction_sha256_float32\':array_sha(p),\'target_sha256_float32\':array_sha(y),\n          \'first_32_predictions_match\':True,\'parameter_count\':sum(q.numel() for q in model.parameters()),**m})\n        del saved,model,p,y\n        torch.cuda.empty_cache()\n    save_json(config[\'output\'],report)\n    print(\'PREFLIGHT PASSED:\',ds,flush=True)\n\ndef summarize(root,references,datasets,seeds):\n    root=Path(root);rows=[];per_gene=[];provenance=[]\n    for ds in datasets:\n        for rec in [r for r in references if r[\'dataset\']==ds and r[\'seed\'] in seeds]:\n            split=json.loads(Path(rec[\'split_path\']).read_text());full,y,genes=load_reference(rec,split)\n            for variant in [\'full\',*VARIANTS]:\n                p=full;path=Path(rec[\'npz\'])\n                if variant!=\'full\':\n                    path=root/\'runs\'/ds/f"{variant}_seed{rec[\'seed\']}"/f"seed{rec[\'seed\']}_test/test_predictions.npz"\n                    if not path.exists():continue\n                    with np.load(path,allow_pickle=False) as z:\n                        if not np.array_equal(z[\'test_indices\'],split[\'test_indices\']):raise ValueError(\'Ablation test indices changed\')\n                        if list(z[\'genes\'].astype(str))!=list(genes):raise ValueError(\'Ablation gene order changed\')\n                        if array_sha(z[\'target\'])!=rec[\'target_sha256_float32\']:raise ValueError(\'Ablation target changed\')\n                        p=z[\'prediction\']\n                met,corr=metrics(p,y)\n                rows.append({\'dataset\':ds,\'variant\':variant,\'seed\':rec[\'seed\'],**met})\n                per_gene.extend({\'dataset\':ds,\'variant\':variant,\'seed\':rec[\'seed\'],\'gene\':g,\'PCC\':float(c)} for g,c in zip(genes,corr))\n                provenance.append({\'dataset\':ds,\'variant\':variant,\'seed\':rec[\'seed\'],\'path\':str(path),\n                                   \'target_sha256_float32\':array_sha(y),\'prediction_sha256_float32\':array_sha(p)})\n    frame=pd.DataFrame(rows);frame.to_csv(root/\'per_seed_test_metrics.csv\',index=False)\n    pd.DataFrame(per_gene).to_csv(root/\'per_gene_test_PCC.csv\',index=False)\n    aggregate=frame.groupby([\'dataset\',\'variant\'])[[\'median_gene_pcc\',\'mse\',\'mse0\',\'ev_zero_percent\']].agg([\'count\',\'mean\',\'std\'])\n    aggregate.columns=[\'_\'.join(c) for c in aggregate.columns]\n    aggregate.reset_index().to_csv(root/\'mean_sample_sd.csv\',index=False)\n    expected=len(datasets)*len(seeds)*(1+len(VARIANTS))\n    save_json(root/\'summary_status.json\',{\'complete\':len(rows)==expected,\'actual_rows\':len(rows),\'expected_rows\':expected,\n        \'definitions\':VARIANTS,\'sd_ddof\':1,\'test_used_for_selection\':False,\n        \'scope\':\'This helper checks the full model and distance control on fixed partitions. Current Table 1 aggregation is provided by scripts/reproduce_tables.py.\'})\n    save_json(root/\'prediction_provenance.json\',provenance)\n    return frame\n\nif __name__==\'__main__\':\n    full_preflight(json.loads(Path(sys.argv[1]).read_text()))\n', 'spagat/attention.py': 'import torch\r\nimport torch.nn as nn\r\nimport torch.nn.functional as F\r\nfrom .embedding import FFN\r\n\r\ndef rho(x):\r\n    return torch.sqrt(F.relu(x)) - torch.sqrt(F.relu(-x))\r\n\r\ndef get_diagonal(x):\n    index = torch.arange(0, x.shape[1], 1)\n    return x[:, index, index, :]\n\n\ndef _normalize_pairwise_tensors(node, edge, embedding, edge_dim=None):\n    """Normalize Embedding\'s initial singleton receiver axis.\n\n    ``Embedding.forward`` supplies pairwise tensors as ``(B, 1, N, E)``: one\n    sender-wise edge representation shared by every receiver.  GRIT attention\n    operates on an explicit receiver/sender representation ``(B, N, N, E)``.\n    Expanding the singleton receiver axis before Q/K--edge arithmetic prevents\n    PyTorch from broadcasting the batch dimension into an unintended\n    ``(B, B, N, N, E)`` tensor.  Later GRIT layers already use the expanded\n    four-dimensional convention.\n    """\n    if node.ndim != 3:\n        raise ValueError(f"node must have shape (B, N, node_dim); got {tuple(node.shape)}.")\n    B, N, _ = node.shape\n\n    def normalize(name, tensor):\n        if tensor.ndim != 4:\n            raise ValueError(\n                f"{name} must have shape (B, 1, N, edge_dim) or (B, N, N, edge_dim); "\n                f"got {tuple(tensor.shape)}."\n            )\n        if tensor.shape[0] != B or tensor.shape[2] != N:\n            raise ValueError(\n                f"{name} batch and sender dimensions must be B={B} and N={N}; "\n                f"got {tuple(tensor.shape)}."\n            )\n        if tensor.shape[1] == 1:\n            tensor = tensor.expand(B, N, N, tensor.shape[-1])\n        elif tensor.shape[1] != N:\n            raise ValueError(\n                f"{name} receiver dimension must be 1 or N={N}; got {tuple(tensor.shape)}."\n            )\n        return tensor\n\n    edge = normalize("edge", edge)\n    embedding = normalize("embedding", embedding)\n    if edge.shape[-1] != embedding.shape[-1]:\n        raise ValueError(\n            "edge and embedding must have the same feature dimension; got "\n            f"{edge.shape[-1]} and {embedding.shape[-1]}."\n        )\n    if edge_dim is not None and edge.shape[-1] != edge_dim:\n        raise ValueError(\n            f"edge and embedding feature dimension must be edge_dim={edge_dim}; "\n            f"got {edge.shape[-1]}."\n        )\n    return edge, embedding\n\r\nclass GRIT_attention(nn.Module):\n    def __init__(self, node_dim, edge_dim, att_dim=8, program_aware=False, routing_mode="program"):\n        super().__init__()\n        self.program_aware = program_aware\n        if routing_mode not in {"program"}:\n            raise ValueError("routing_mode must be \'program\'.")\n        self.routing_mode = routing_mode\n        self.node_dim = node_dim\n        self.base_edge_dim = edge_dim\n        self.edge_dim = edge_dim * 2\n        self.W_Q = nn.Linear(node_dim, self.edge_dim * att_dim)\r\n        self.W_K = nn.Linear(node_dim, self.edge_dim * att_dim)\r\n        self.W_V = nn.Linear(node_dim, node_dim)\r\n        self.W_Ew = nn.Linear(edge_dim, edge_dim, bias=False)\r\n        self.W_Eb = nn.Linear(edge_dim, edge_dim, bias=False)\r\n        self.W_En = nn.Linear(edge_dim, node_dim)\r\n        self.W_A = nn.Linear(edge_dim, 1, bias=False)\r\n        self.W_No = nn.Linear(node_dim, node_dim)\n        self.W_Eo = nn.Linear(edge_dim, edge_dim, bias=False)\n\n        # Program tokens live in the common edge-feature space (edge_dim).  They\n        # are supplied by a parent model so that token k has the same biological\n        # identity in every graph layer; these projections remain layer-specific.\n        if self.program_aware:\n            self.W_program_edge = nn.Linear(edge_dim, edge_dim, bias=False)\n            self.W_program_receiver = nn.Linear(node_dim, edge_dim, bias=False)\n            self.W_program_gate = nn.Linear(node_dim, edge_dim, bias=False)\n\n    def forward(self, x, program_tokens=None, program_edge_mask=None):\n        """Run edge-aware attention, optionally routed through communication programs.\n\n        Args:\n            x: ``[node, edge, embedding]`` with node ``(B, N, node_dim)`` and\n                broadcastable edge/embedding tensors ``(B, 1 or N, N, edge_dim)``.\n            program_tokens: Optional shared token bank ``(K, edge_dim)``.  A\n                token is a globally named latent communication program; the\n                same bank should be passed to every program-aware GRIT layer.\n            program_edge_mask: Optional inference-only ``(B, N, N, K)`` mask.\n                Zero entries remove sender edges from a program\'s softmax.\n\n        Returns:\n            ``[node_updated, edge_updated, program_state]``.  ``program_state``\n            is ``None`` in baseline mode, preserving the original computation.\n        """\n        node, edge, embedding = x\n        B, N, C = node.shape\n\n        # Always normalize the initial Embedding output, including the legacy\n        # path.  The routing mode must not determine whether Q/K arithmetic is\n        # performed on a correctly shaped pairwise edge tensor.\n        edge, embedding = _normalize_pairwise_tensors(\n            node, edge, embedding, edge_dim=self.base_edge_dim\n        )\n\n        if program_tokens is not None and not self.program_aware:\n            raise ValueError(\n                "program_tokens were provided, but this GRIT_attention was "\n                "constructed with program_aware=False."\n            )\n        if program_tokens is not None and program_tokens.ndim != 2:\n            raise ValueError(\n                "program_tokens must have shape (n_programs, edge_dim); got "\n                f"{tuple(program_tokens.shape)}."\n            )\n        if program_tokens is not None and program_tokens.shape[1] != self.base_edge_dim:\n            raise ValueError(\n                "program_tokens must have shape (n_programs, edge_dim), where "\n                f"edge_dim={self.base_edge_dim}; got {tuple(program_tokens.shape)}."\n            )\n        if program_edge_mask is not None and program_tokens is None:\n            raise ValueError("program_edge_mask requires program-aware routing tokens.")\n\n        Q = self.W_Q(node).reshape(B, N, -1, self.edge_dim).permute(0, 3, 1, 2)\n        K = self.W_K(node).reshape(B, N, -1, self.edge_dim).permute(0, 3, 2, 1)\r\n        QK = (Q @ K).permute(0, 2, 3, 1)\r\n        edge = F.gelu(rho((QK[:, :, :, :self.edge_dim // 2]) * self.W_Ew(edge)) + self.W_Eb(edge) + QK[:, :, :, self.edge_dim // 2:] + embedding)\r\n        V = self.W_V(node)\n\n        if program_tokens is None:\n            # This is the original GRIT message-passing path, kept numerically\n            # unchanged when no shared program bank is supplied.\n            alpha = self.W_A(edge).squeeze(dim=-1)\n            alpha = F.softmax(alpha, dim=-1)\n            node = alpha @ V + self.W_En(torch.sum(edge * alpha.unsqueeze(dim=-1), dim=-2))\n            node = self.W_No(node)\n            edge = self.W_Eo(edge)\n            return [node, edge, None]\n\n        edge_program = self.W_program_edge(edge)\n        receiver_program = self.W_program_receiver(node)\n        program_scores = torch.einsum(\n            "bije,bie,ke->bijk", edge_program, receiver_program, program_tokens\n        ) / (self.base_edge_dim ** 0.5)\n        program_attention = None\n        if program_edge_mask is not None:\n            expected_shape = (B, N, N, program_tokens.shape[0])\n            if tuple(program_edge_mask.shape) != expected_shape:\n                raise ValueError(\n                    f"program_edge_mask must have shape {expected_shape}; got "\n                    f"{tuple(program_edge_mask.shape)}."\n                )\n            allowed = program_edge_mask.to(device=program_scores.device, dtype=torch.bool)\n            if not torch.all(allowed.any(dim=2)):\n                raise ValueError("program_edge_mask removes every sender for at least one receiver/program.")\n            # This is inference-time graph intervention: masked edges receive\n            # exactly zero attention and remaining edges are renormalized by\n            # the existing neighbor softmax.\n            program_scores = program_scores.masked_fill(~allowed, float("-inf"))\n            program_attention = F.softmax(program_scores, dim=2)\n        elif program_attention is None:\n            program_attention = F.softmax(program_scores, dim=2)\n\n        # Aggregate node and edge evidence independently for each program:\n        # m[i, k] = sum_j alpha[i, j, k] * message[i, j].\n        node_messages = torch.einsum("bijk,bjd->bikd", program_attention, V)\n        edge_messages = torch.einsum("bijk,bije->bike", program_attention, edge)\n        program_messages = node_messages + self.W_En(edge_messages)\n\n        # Receiver-specific, non-negative gates fuse the K program messages back\n        # into the original node_dim, keeping the downstream tensor shape intact.\n        gate_scores = torch.einsum(\n            "bin,kn->bik", self.W_program_gate(node), program_tokens\n        ) / (self.base_edge_dim ** 0.5)\n        program_activations = F.softplus(gate_scores)\n        node = torch.sum(program_messages * program_activations.unsqueeze(dim=-1), dim=2)\n        node = self.W_No(node)\n        edge = self.W_Eo(edge)\n\n        program_state = {\n            "attention": program_attention,\n            "messages": program_messages,\n            "activations": program_activations,\n        }\n        return [node, edge, program_state]\n\r\nclass DegScaler(nn.Module):\r\n    def __init__(self, node_dim):\r\n        super().__init__()\r\n        scaler = (2 / node_dim) ** 0.5\r\n        self.theta1 = nn.Parameter(torch.randn(node_dim) * scaler)\r\n        self.theta2 = nn.Parameter(torch.randn(node_dim) * scaler)\r\n\r\n    def forward(self, node, degree):\r\n        node = self.theta1 * node + torch.log(degree + 1).unsqueeze(dim=-1) * node * self.theta2\r\n        return node\r\n\r\nclass Multi_Head_Attention(nn.Module):\r\n    def __init__(self, node_dim, edge_dim, num_heads, att_dim=8):\r\n        super().__init__()\r\n        self.attentions = nn.ModuleList([\r\n            GRIT_attention(node_dim, edge_dim, att_dim) for _ in range(num_heads)\r\n        ])\r\n        self.W_hn = nn.Parameter(torch.ones(num_heads, 1) / num_heads)\r\n        self.W_he = nn.Parameter(torch.ones(num_heads, 1) / num_heads)\r\n\r\n    def forward(self, x):\r\n        results = [attentioni(x) for attentioni in self.attentions]\r\n        node = (torch.stack([tmp[0] for tmp in results], dim=-1) @ self.W_hn).squeeze(dim=-1)\r\n        edge = torch.stack([tmp[1] for tmp in results], dim=-1) @ self.W_he\r\n        edge = edge.squeeze(dim=-1)\r\n        return [node, edge, x[2]]\r\n\r\nclass GRIT_encoder(nn.Module):\n    def __init__(self, node_dim, edge_dim, num_heads, att_dim=8, program_aware=False, routing_mode="program"):\n        super().__init__()\n        self.program_aware = program_aware\n        self.edge_dim = edge_dim\n        self.routing_mode = routing_mode\n        self.attentions = Multi_Head_Attention(node_dim, edge_dim, num_heads, att_dim)\n        if program_aware:\n            # Keep Multi_Head_Attention unchanged for baseline callers, but replace\n            # its heads with token-aware GRIT attention modules for this encoder.\n            # The shared tokens themselves are supplied to forward(), not owned by\n            # a layer, so program index k remains consistent across GNN layers.\n            self.attentions.attentions = nn.ModuleList([\n                GRIT_attention(node_dim, edge_dim, att_dim, program_aware=True, routing_mode=routing_mode)\n                for _ in range(num_heads)\n            ])\n        self.FFN = FFN(node_dim)\n        self.ln1 = nn.LayerNorm(node_dim)\n        self.ln2 = nn.LayerNorm(node_dim)\n        self.ln_edge = nn.LayerNorm(edge_dim)\n\n    def forward(self, x, program_tokens=None, program_edge_mask=None):\n        """Encode one graph layer, optionally with shared program-aware routing.\n\n        Without ``program_tokens``, this follows the original GRIT encoder path\n        and returns ``[node, edge, distance_embedding]``.  With a shared token\n        bank, each head returns program-specific routing state; the encoder\n        combines it across heads and returns it as a fourth element.\n        """\n        node, edge, distance = x\n        # Keep the residual edge tensor in the same four-dimensional convention\n        # as GRIT_attention.  This is essential for the baseline residual add.\n        edge, distance = _normalize_pairwise_tensors(\n            node, edge, distance, edge_dim=self.edge_dim\n        )\n        x = [node, edge, distance]\n        if program_tokens is None:\n            # Preserve the existing multi-head aggregation and output contract.\n            x = self.attentions(x)\n            edge = self.ln_edge(edge + x[1])\n            node = self.ln1(node + x[0])\n            node = self.ln2(node + self.FFN(node))\n            return [node, edge, distance]\n\n        if not self.program_aware:\n            raise ValueError(\n                "program_tokens were provided, but this GRIT_encoder was "\n                "constructed with program_aware=False."\n            )\n\n        # Multi_Head_Attention\'s baseline forward intentionally remains unchanged.\n        # Here, route the same shared program tokens through each token-aware head\n        # and use its existing learned head weights for node and edge aggregation.\n        results = [\n            attentioni(x, program_tokens, program_edge_mask=program_edge_mask)\n            for attentioni in self.attentions.attentions\n        ]\n        node_attention = (\n            torch.stack([result[0] for result in results], dim=-1) @ self.attentions.W_hn\n        ).squeeze(dim=-1)\n        edge_attention = torch.stack([result[1] for result in results], dim=-1) @ self.attentions.W_he\n        edge_attention = edge_attention.squeeze(dim=-1)\n\n        # Combine per-head routing metadata with normalized head weights.  This\n        # yields one program axis K shared by all heads and preserves alpha\'s\n        # interpretation as a neighbor distribution for each receiver/program.\n        head_weights = F.softmax(self.attentions.W_hn.squeeze(dim=-1), dim=0)\n        program_state = {\n            key: torch.sum(\n                torch.stack([result[2][key] for result in results], dim=-1)\n                * head_weights,\n                dim=-1,\n            )\n            for key in ("attention", "messages", "activations")\n        }\n\n        # Pairwise tensors were normalized at entry, so the residual update is\n        # always performed in the explicit ``(B, N, N, E)`` convention.\n        edge = self.ln_edge(edge + edge_attention)\n        node = self.ln1(node + node_attention)\n        node = self.ln2(node + self.FFN(node))\n        return [node, edge, distance, program_state]\n\r\nclass GRIT_encoder_last_layer(nn.Module):\r\n    def __init__(self, node_dim, in_node, edge_dim, node_dim_small=16, att_dim=8):\r\n        super().__init__()\r\n        self.node_dim_small = node_dim_small\r\n        self.in_node = in_node\r\n        self.edge_dim = edge_dim * 2\r\n        self.att_dim = att_dim\r\n        self.W_Q = nn.Linear(node_dim, self.edge_dim * att_dim)\r\n        self.W_K = nn.Linear(node_dim, self.edge_dim * att_dim)\r\n        self.W_Ew = nn.Linear(edge_dim, edge_dim, bias=False)\r\n        self.W_Eb = nn.Linear(edge_dim, edge_dim, bias=False)\r\n        self.W_En = nn.Linear(edge_dim, node_dim)\r\n        self.W_A = nn.Linear(edge_dim, in_node, bias=False)\r\n        self.node_transform = nn.Linear(node_dim, in_node)\r\n        self.edge_transform = nn.Linear(edge_dim, in_node)\r\n        self.head = FFN(in_node, in_dim=in_node * 2, out_dim=in_node)\r\n        self.head2 = nn.Sequential(nn.LayerNorm(in_node), FFN(in_node))\r\n        self.scaler = 2 / (node_dim ** 0.5)\r\n\r\n    def forward(self, x):\r\n        node, edge, embedding = x\r\n        B, N, C = node.shape\r\n        Q = self.W_Q(node[:, 0:1, :]).reshape(B, 1, self.att_dim, self.edge_dim).permute(0, 3, 1, 2)\r\n        K = self.W_K(node).reshape(B, N, self.att_dim, self.edge_dim).permute(0, 3, 2, 1)\r\n        QK = (Q @ K).permute(0, 2, 3, 1)\r\n        edge = edge[:, 0:1, :, :]\r\n        edge = F.gelu(rho((QK[:, :, :, :self.edge_dim // 2]) * self.W_Ew(edge)) + self.W_Eb(edge) + QK[:, :, :, self.edge_dim // 2:] + embedding[:, 0:1, :, :])\r\n        edge = edge[:, 0, 1:, :]\r\n        alphas = F.softmax(self.W_A(edge).permute(0, 2, 1), dim=-1).permute(0, 2, 1)\r\n        edge = self.edge_transform(edge) * alphas\r\n        node = self.node_transform(node[:, 1:, :]) * alphas\r\n        tmp = self.head(torch.concat([edge, node], dim=-1))\r\n        node = torch.sum(tmp, dim=-2) * self.scaler\r\n        return [node, [tmp.permute(0, 2, 1).unsqueeze(dim=-2), edge.permute(0, 2, 1).unsqueeze(dim=-2)]]\r\n\r\nclass ProgramLastLayer(nn.Module):\r\n    def __init__(self, node_dim, in_node, edge_dim, n_programs=8, node_dim_small=16, att_dim=8):\r\n        super().__init__()\r\n        self.node_dim_small = node_dim_small\r\n        self.in_node = in_node\r\n        self.n_programs = n_programs\r\n        self.edge_dim = edge_dim * 2\r\n        self.att_dim = att_dim\r\n        self.W_Q = nn.Linear(node_dim, self.edge_dim * att_dim)\r\n        self.W_K = nn.Linear(node_dim, self.edge_dim * att_dim)\r\n        self.W_Ew = nn.Linear(edge_dim, edge_dim, bias=False)\r\n        self.W_Eb = nn.Linear(edge_dim, edge_dim, bias=False)\r\n        self.W_A = nn.Linear(edge_dim, in_node * n_programs, bias=False)\r\n        self.node_transform = nn.Linear(node_dim, in_node)\r\n        self.edge_transform = nn.Linear(edge_dim, in_node)\r\n        self.head = FFN(in_node, in_dim=in_node * 2, out_dim=in_node)\r\n        self.scaler = 2 / (node_dim ** 0.5)\r\n\r\n    def forward(self, x):\r\n        node, edge, embedding = x\r\n        B, N, C = node.shape\r\n        Q = self.W_Q(node[:, 0:1, :]).reshape(B, 1, self.att_dim, self.edge_dim).permute(0, 3, 1, 2)\r\n        K_mat = self.W_K(node).reshape(B, N, self.att_dim, self.edge_dim).permute(0, 3, 2, 1)\r\n        QK = (Q @ K_mat).permute(0, 2, 3, 1)\r\n        edge = edge[:, 0:1, :, :]\r\n        edge = F.gelu(\r\n            rho((QK[:, :, :, :self.edge_dim // 2]) * self.W_Ew(edge))\r\n            + self.W_Eb(edge)\r\n            + QK[:, :, :, self.edge_dim // 2:]\r\n            + embedding[:, 0:1, :, :]\r\n        )\r\n        edge = edge[:, 0, 1:, :]\r\n        alphas_all = self.W_A(edge)\r\n        alphas_all = alphas_all.reshape(B, N - 1, self.in_node, self.n_programs)\r\n        alphas_all = alphas_all.permute(0, 3, 2, 1)\r\n        alphas_all = F.softmax(alphas_all, dim=-1)\r\n        alphas_all = alphas_all.permute(0, 3, 1, 2)\r\n        edge_feat = self.edge_transform(edge)\r\n        node_feat = self.node_transform(node[:, 1:, :])\r\n        program_outputs = []\r\n        program_influences = []\r\n        for k in range(self.n_programs):\r\n            alpha_k = alphas_all[:, :, k, :]\r\n            edge_k = edge_feat * alpha_k\r\n            node_k = node_feat * alpha_k\r\n            tmp_k = self.head(torch.cat([edge_k, node_k], dim=-1))\r\n            output_k = torch.sum(tmp_k, dim=1) * self.scaler\r\n            program_outputs.append(output_k)\r\n            program_influences.append(tmp_k)\r\n        y_pred = torch.stack(program_outputs, dim=0).sum(dim=0)\r\n        total_influence = torch.stack(program_influences, dim=0).sum(dim=0)\r\n        influence_for_compat = total_influence.permute(0, 2, 1).unsqueeze(dim=-2)\r\n        program_info = {\r\n            \'per_program_influence\': torch.stack(program_influences, dim=0),\r\n            \'per_program_output\': torch.stack(program_outputs, dim=0),\r\n            \'per_program_attention\': alphas_all,\r\n        }\r\n        return [y_pred, [influence_for_compat, program_info]]\n', 'spagat/preprocess.py': 'import torch\r\n\r\ndef process(x,judge_distance=28):\r\n    \'\'\'\r\n    :param x:\r\n        "x": exp,\r\n        "y": y,\r\n        "ligand":ligand,\r\n        "receptor":receptor,\r\n        "cell_types": cell_types,\r\n        "morphology": volumes,\r\n        "position_x": centerx,\r\n        "position_y": centery\r\n    :return:\r\n        "x": b,n,c1\r\n        "interactions": b,n,n,c2\r\n        "mask": b,n,n\r\n        "cell_types": b,n\r\n        "distance_matrix": b,n,n,c3\r\n        "y": b,c\r\n        "morphology": b,n,c4\r\n    \'\'\'\r\n    dx = x["position_x"][:, 0:1] - x["position_x"]\r\n    dy = x["position_y"][:, 0:1] - x["position_y"]\r\n    distances=torch.sqrt(torch.square(dx)+torch.square(dy))\r\n\r\n    \'\'\'mask=torch.where(distances<judge_distance,\r\n                     torch.ones_like(distances,device=distances.device),\r\n                     torch.zeros_like(distances,device=distances.device))\'\'\'\r\n    distance_matrix = torch.stack([1 / (distances + 1), 1 / (torch.sqrt(distances) + 1),\r\n                                   1 / (1 + torch.square(distances)), torch.exp(-distances),\r\n                                   torch.exp(-torch.square(distances))], dim=-1)\r\n    return {\r\n        "x":x["x"],\r\n        "type_exp": x["type_exp"],\r\n        "y":x["y"],\r\n        "cell_types": x["cell_types"],\r\n        "distance_matrix":distance_matrix\r\n    }', 'spagat/embedding.py': 'import torch\r\nimport torch.nn as nn\r\nimport torch.nn.functional as F\r\nfrom . import preprocess\r\n\r\nclass FFN(nn.Module):\r\n    def __init__(self,dim,in_dim=None,out_dim=None,bias=True):\r\n        super().__init__()\r\n        if in_dim is None:\r\n            in_dim=dim\r\n        if out_dim is None:\r\n            out_dim=dim\r\n\r\n        self.linear1=nn.Linear(in_dim,dim*4,bias=bias)\r\n        self.linear2=nn.Linear(dim*4,out_dim,bias=bias)\r\n        self.act=nn.GELU()\r\n\r\n    def forward(self,x):\r\n        return self.linear2(self.act(self.linear1(x)))\r\n\r\ndef group_strings_by_numbers(strings, numbers):\r\n    """\r\n    Group a list of strings based on unique numbers and return as two ordered lists.\r\n\r\n    :param strings: List of strings to be grouped.\r\n    :param numbers: List of numbers indicating the grouping.\r\n    :return: Two lists, one with unique numbers and the other with corresponding grouped strings.\r\n    """\r\n    numbers=numbers.copy()[:len(strings)]\r\n    unique_numbers = sorted(set(numbers))\r\n    grouped_strings = [[] for _ in unique_numbers]\r\n\r\n    for string, number in zip(strings, numbers):\r\n        index = unique_numbers.index(number)\r\n        grouped_strings[index].append(string)\r\n\r\n    return unique_numbers, grouped_strings\r\n\r\nclass Embedding(nn.Module):\r\n    def __init__(self,genes,ligands_info,node_dim,edge_dim,use_cell_type_embedding=True):\r\n        super().__init__()\r\n        self.use_cell_type_embedding=use_cell_type_embedding\r\n        self.ligands_info=ligands_info\r\n\r\n        # for scaling the data\r\n        #self.exp_scaler=nn.Parameter(torch.ones((len(genes))))\r\n\r\n        # for node\r\n        self.cell_encoder1 = FFN(node_dim, in_dim=len(genes))\r\n        self.cell_encoder2 = nn.Embedding(29,node_dim)\r\n\r\n        # for distance\r\n        self.distance_scaler=nn.Sequential(FFN(edge_dim,out_dim=len(ligands_info[0]),in_dim=5),nn.Sigmoid())\r\n        self.distance_encoder=FFN(edge_dim,out_dim=len(ligands_info[0]),in_dim=5)\r\n        self.distance_embedding = nn.Sequential(FFN(edge_dim, in_dim=5), nn.Sigmoid())\r\n\r\n        # for edge\r\n        self.ligands_encoder=nn.Linear(len(ligands_info[0]),edge_dim)\r\n        self.edge_encoder=nn.Linear(len(ligands_info[0])+edge_dim,edge_dim,bias=False)\r\n        self.scaler=edge_dim**0.5\r\n\r\n        # calculate ligand scores\r\n        self.ligands_index=[]\r\n        for i in range(len(self.ligands_info[0])):\r\n            ligandi = self.ligands_info[0][i]\r\n            ligand_groupi = self.ligands_info[1][i]\r\n            uniquei, groupi = group_strings_by_numbers(ligandi, ligand_groupi)\r\n            groupi=[torch.LongTensor([genes.index(groupi[j][k]) for k in range(len(groupi[j]))]) for j in range(len(groupi))]\r\n            self.ligands_index.append(groupi)\r\n\r\n    def forward(self, x, ligand_feature_mask=None):\r\n        \'\'\'\r\n        :param x:\r\n        x["x"]--b,n,in_node\r\n        x["distance_matrix"]--b,5,n,n\r\n        x["cell_types"]--long,b,n\r\n        x["type_exp"]--b,n,in_node\r\n        :return:\r\n        node_features:b,n,node_dim\r\n        edge1:b,n,n,edge_dim\r\n        edge2:b,n,n,edge_dim+5\r\n        rw: b,n,n,num_hops+5\r\n        \'\'\'\r\n        x=preprocess.process(x)\n        if getattr(self, "component_ablation", "full") == "no_distance":\n            x["distance_matrix"] = torch.zeros_like(x["distance_matrix"])\r\n\r\n        # node features\r\n        #scaler=torch.square(self.exp_scaler)\r\n        exp=x["type_exp"]+x["x"]\r\n        exp=torch.where(x["cell_types"].unsqueeze(dim=-1)==x["cell_types"][:,0:1].unsqueeze(dim=-1),x["type_exp"],exp)\r\n        exp=exp#*scaler\r\n        node_features1=self.cell_encoder1(exp)\r\n        node_features2 = self.cell_encoder1(x["type_exp"])#*scaler)\r\n        node_features=torch.where(x["cell_types"].unsqueeze(dim=-1)==x["cell_types"][:,0:1].unsqueeze(dim=-1),node_features2,node_features1)\r\n\r\n        if self.use_cell_type_embedding:\r\n            node_features=node_features+self.cell_encoder2(x["cell_types"])\r\n\r\n        # calculate ligand scores\r\n        B,N,C=node_features.shape\r\n        data_ligand=[]\r\n        for i in range(len(self.ligands_info[0])):\r\n            data_ligandi=torch.ones((B,N),device=node_features.device)\r\n            for j in range(len(self.ligands_index[i])):\r\n                data_ligandi = data_ligandi * torch.mean(exp[:,:,self.ligands_index[i][j]],dim=-1)\r\n            data_ligandi=torch.pow(data_ligandi,exponent=1 / len(self.ligands_index[i]))\r\n            data_ligand.append(data_ligandi)\r\n        data_ligand=torch.stack(data_ligand,dim=-1)\n\r\n        # distance scaler\r\n        distance_scaler = self.distance_scaler(x["distance_matrix"])\r\n\r\n        # ligand score scaled by distance\r\n        ligand_score=distance_scaler*data_ligand\r\n        if ligand_feature_mask is not None:\r\n            ligand_score = ligand_score * ligand_feature_mask\r\n        edge_ligand=self.ligands_encoder(ligand_score)\r\n\r\n        # edge features\r\n        edge_distance=self.distance_encoder(x["distance_matrix"])\r\n        edges = self.edge_encoder(torch.concat([edge_distance,edge_ligand],dim=-1))*self.scaler\r\n\r\n        # distance embedding\r\n        embedding = self.distance_embedding(x["distance_matrix"])\r\n        return [node_features,edges.unsqueeze(dim=1),embedding.unsqueeze(dim=1)]\r\n\r\n\r\n\r\n', 'spagat/__init__.py': '"""\r\nSpaGAT package.\r\n"""\r\n\r\nfrom .gene_program_model import SpaGP, SpaGP_Loss\r\nfrom .dataloader import SPAGAT_dataset\r\n\r\n__all__ = [\r\n    "SpaGP",\r\n    "SpaGP_Loss",\r\n    "SPAGAT_dataset",\r\n]', 'spagat/gene_program_model.py': '"""\nSpatial Gene Program Model (SpaGP)\n\nCore idea:\n    y_i = μ(cell_type_i) + Σ_k g_k(z_i) · p_k + ε\n\nwhere:\n    - μ(cell_type) is cell-type baseline expression (from type_exp)\n    - p_k ∈ R^genes is the k-th gene program (global, shared, orthogonal)\n    - g_k(z_i) ∈ R+ is scalar activation of program k for cell i (non-negative)\n    - z_i ∈ R^d is spatial context embedding from neighborhood via graph encoder\n"""\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .attention import GRIT_encoder\nfrom .embedding import Embedding, FFN\n\n\nclass GeneProgramHead(nn.Module):\n    """\n    The program-based prediction head.\n    \n    Given spatial context embedding z_i (from graph encoder),\n    predicts gene expression as:\n        y_i = baseline_i + Σ_k g_k(z_i) · p_k\n    """\n    def __init__(self, node_dim, n_genes, n_programs=16, n_neighbors=50,\n                 program_aware=False, message_decoder="free"):\n        super().__init__()\n        self.n_genes = n_genes\n        self.n_programs = n_programs\n        self.n_neighbors = n_neighbors\n        self.program_aware = program_aware\n        self.message_decoder = message_decoder\n        if message_decoder not in {"free", "program_aligned"}:\n            raise ValueError("message_decoder must be \'free\' or \'program_aligned\'.")\n        \n        # ===== Gene Programs p_k ∈ R^{K × genes} =====\n        # Initialized with Xavier, will be orthogonalized during training via loss penalty\n        self.programs = nn.Parameter(torch.empty(n_programs, n_genes))\n        nn.init.xavier_uniform_(self.programs)\n        \n        # ===== Context Aggregator: aggregate neighborhood into z_i =====\n        # Attention-weighted aggregation of neighbor node features → z_i\n        self.context_query = nn.Linear(node_dim, node_dim)\n        self.context_key = nn.Linear(node_dim, node_dim)\n        self.context_proj = nn.Sequential(\n            nn.Linear(node_dim, node_dim),\n            nn.GELU(),\n            nn.Linear(node_dim, node_dim)\n        )\n        self.scale = node_dim ** 0.5\n        \n        # ===== Activation Network: z_i → g(z_i) ∈ R+^K =====\n        self.activation_net = nn.Sequential(\n            nn.Linear(node_dim, node_dim),\n            nn.GELU(),\n            nn.Linear(node_dim, n_programs),\n            nn.Softplus()  # ensures g_k ≥ 0\n        )\n        \n        # ===== Baseline: cell-type expression → gene-level baseline =====\n        self.baseline_head = nn.Sequential(\n            nn.Linear(n_genes, n_genes),\n            nn.GELU(),\n            nn.Linear(n_genes, n_genes)\n        )\n\n        # Only program-aware SpaGP uses this decoder branch.  Keeping it absent\n        # from the legacy construction preserves existing checkpoint layouts.\n        if program_aware:\n            if message_decoder == "free":\n                self.message_to_genes = nn.Linear(node_dim, n_genes)\n            else:\n                # A shared scalar decoder yields q_k from each program message\n                # m_k.  Its gene effect is constrained to that program\'s P_k.\n                self.message_to_program = nn.Linear(node_dim, 1)\n    \n    def forward(self, node_features, type_exp_center, encoder_program_state=None, program_mask=None):\n        """\n        Args:\n            node_features: (B, N, node_dim) — encoder output for center + neighbors\n            type_exp_center: (B, n_genes) — cell-type average expression for center cell\n            encoder_program_state: Optional state returned by a program-aware\n                GRIT encoder.  When supplied, its center-cell activations replace\n                this head\'s legacy context/activation inference.\n            program_mask: Optional inference-only ``(K,)`` 0/1 mask. It is\n                applied only to program-specific decoder contributions; the\n                shared cell-type baseline remains unchanged.\n        \n        Returns:\n            y_pred: (B, n_genes)\n            program_info: dict with activations and programs for downstream analysis\n        """\n        B, N, C = node_features.shape\n        \n        if encoder_program_state is None:\n            # Legacy decoder path: retain the original head computation and\n            # parameter layout so existing SpaGP checkpoints remain usable.\n            center = node_features[:, 0:1, :]\n            neighbors = node_features[:, 1:, :]\n            Q = self.context_query(center)\n            K = self.context_key(neighbors)\n            attn = F.softmax((Q @ K.transpose(-1, -2)) / self.scale, dim=-1)\n            z_i = self.context_proj(attn @ neighbors).squeeze(1)\n            g = self.activation_net(z_i)\n            program_attention = None\n            program_messages = None\n        else:\n            # Program-aware decoder path: graph message passing has already\n            # produced non-negative activations for every node/program pair.\n            # The center-cell row is the activation vector used to decode genes.\n            g = encoder_program_state[\'activations\'][:, 0, :]\n            z_i = node_features[:, 0, :]\n            program_attention = encoder_program_state[\'attention\'][:, 0, 1:, :]\n            program_messages = encoder_program_state[\'messages\'][:, 0, :, :]\n\n            # Keep the historical (B, N-1) attn_weights field as an activation-\n            # weighted summary of the full program-specific attention tensor.\n            gate_weights = g / (torch.sum(g, dim=-1, keepdim=True) + 1e-8)\n            attn = torch.sum(program_attention * gate_weights.unsqueeze(dim=1), dim=-1).unsqueeze(dim=1)\n        \n        # --- Step 3: Normalize programs (unit norm per program) ---\n        programs_normalized = F.normalize(self.programs, dim=-1)  # (K, genes)\n        if program_mask is not None:\n            program_mask = torch.as_tensor(\n                program_mask, device=programs_normalized.device, dtype=programs_normalized.dtype\n            )\n            if program_mask.ndim != 1 or program_mask.shape[0] != self.n_programs:\n                raise ValueError(\n                    f"program_mask must have shape ({self.n_programs},); got {tuple(program_mask.shape)}."\n                )\n        mask_is_identity = program_mask is None or bool(torch.all(program_mask == 1))\n        \n        # --- Step 4: Program contribution ---\n        # Σ_k g_k(z_i) · p_k = g @ P\n        # Keep the default expression byte-for-byte on the existing path.  For\n        # an intervention, mask each program\'s gene-program contribution before\n        # summing across k; no re-normalization occurs.\n        masked_g = g if mask_is_identity else g * program_mask.unsqueeze(0)\n        program_output = masked_g @ programs_normalized   # (B, genes)\n\n        if encoder_program_state is None:\n            message_output = None\n            message_coefficients = None\n        else:\n            if not self.program_aware:\n                raise RuntimeError(\n                    "GeneProgramHead received encoder program state but was not "\n                    "constructed with program_aware=True."\n                )\n            if self.message_decoder == "free":\n                # Legacy program-aware ablation: a flexible shared gene decoder\n                # consumes the gate-weighted sum of program messages.\n                if mask_is_identity:\n                    fused_program_message = torch.sum(\n                        program_messages * gate_weights.unsqueeze(dim=-1), dim=1\n                    )\n                    message_output = self.message_to_genes(fused_program_message)\n                else:\n                    # Distribute the free decoder\'s shared bias by the original\n                    # program weights: sum_k w_k (W m_k + b) equals the legacy\n                    # W sum_k(w_k m_k) + b when every mask is one, while an\n                    # all-zero mask removes the complete program message path.\n                    message_weights = gate_weights * program_mask.unsqueeze(0)\n                    message_components = F.linear(\n                        program_messages, self.message_to_genes.weight, self.message_to_genes.bias\n                    )\n                    message_output = torch.sum(message_components * message_weights.unsqueeze(dim=-1), dim=1)\n                message_coefficients = None\n            else:\n                # Program-aligned decoder: q_k is derived from m_k, then its\n                # contribution is restricted to the corresponding gene program\n                # P_k rather than passing through an unrestricted gene decoder.\n                message_coefficients = self.message_to_program(program_messages).squeeze(dim=-1)\n                masked_coefficients = (\n                    message_coefficients if mask_is_identity\n                    else message_coefficients * program_mask.unsqueeze(0)\n                )\n                message_output = masked_coefficients @ programs_normalized\n        \n        # --- Step 5: Baseline from cell-type expression ---\n        baseline = self.baseline_head(type_exp_center)  # (B, genes)\n        \n        # --- Step 6: Final prediction ---\n        y_pred = baseline + program_output\n        if message_output is not None:\n            y_pred = y_pred + message_output\n        \n        program_info = {\n            \'activations\': g,                    # (B, K) — program activation per cell\n            \'programs\': programs_normalized,      # (K, genes) — gene program vectors\n            \'context\': z_i,                       # (B, C) — spatial context embedding\n            \'attn_weights\': attn.squeeze(1),      # (B, N-1) — neighbor attention weights\n            \'baseline\': baseline,                 # (B, genes) — cell-type baseline\n            \'program_output\': program_output      # (B, genes) — program contribution\n        }\n        if program_attention is not None:\n            program_info[\'program_attention\'] = program_attention\n            program_info[\'program_messages\'] = program_messages\n            program_info[\'program_message_output\'] = message_output\n            program_info[\'program_message_coefficients\'] = message_coefficients\n        if program_mask is not None:\n            program_info[\'program_mask\'] = program_mask\n        \n        return y_pred, program_info\n\n\nclass SpaGP(nn.Module):\n    """\n    Spatial Gene Program Model.\n    """\n    def __init__(self, genes, ligands_info, node_dim=256, edge_dim=48, num_heads=2,\n                 n_layers=1, att_dim=8, use_cell_type_embedding=True,\n                 n_programs=16, program_aware=False, program_token_dim=32,\n                 message_decoder="free", routing_mode="program"):\n        super().__init__()\n        self.genes = genes\n        self.n_programs = n_programs\n        self.program_aware = program_aware\n        self.program_token_dim = program_token_dim\n        self.message_decoder = message_decoder\n        self.routing_mode = routing_mode\n\n        if program_aware and n_layers < 1:\n            raise ValueError("SpaGP program-aware mode requires at least one GRIT encoder layer.")\n        \n        self.embeddings = Embedding(genes, ligands_info, node_dim, edge_dim,\n                                    use_cell_type_embedding=use_cell_type_embedding)\n        self.encoders = nn.ModuleList([\n            GRIT_encoder(node_dim, edge_dim, num_heads, att_dim, program_aware=program_aware, routing_mode=routing_mode)\n            for i in range(n_layers)  \n        ])\n\n        if program_aware:\n            # The learned bank is intentionally in a separate latent space P.\n            # Its shared projection supplies (K, edge_dim) tokens required by\n            # every GRIT layer, keeping program identity fixed across depth.\n            self.program_tokens = nn.Parameter(torch.empty(n_programs, program_token_dim))\n            nn.init.xavier_uniform_(self.program_tokens)\n            self.program_token_projection = nn.Linear(program_token_dim, edge_dim, bias=False)\n\n        self.program_head = GeneProgramHead(\n            node_dim=node_dim,\n            n_genes=len(genes),\n            n_programs=n_programs,\n            program_aware=program_aware,\n            message_decoder=message_decoder,\n        )\n    \n    def forward(self, x, program_mask=None, program_edge_mask=None,ligand_feature_mask=None):\n        """\n        Args:\n            x: dict from dataloader with keys:\n                "x": (B, N, genes), "type_exp": (B, N, genes),\n                "cell_types": (B, N), "position_x": (B, N), "position_y": (B, N),\n                "y": (B, genes)\n        \n        Returns:\n            y_pred: (B, genes)\n            program_info: dict with activations, programs, context, etc.\n        """\n        # Center cell\'s type expression (for baseline)\n        type_exp_center = x["type_exp"][:, 0, :]  # (B, genes)\n        encoded = self.embeddings(x,ligand_feature_mask=ligand_feature_mask)   # [node, edge, distance]\n        if self.program_aware:\n            program_tokens = self.program_token_projection(self.program_tokens)\n            final_program_state = None\n            for encoder in self.encoders:\n                # Program state is analysis metadata; only graph tensors are\n                # forwarded to the next GRIT layer.\n                encoded = encoder(\n                    encoded[:3], program_tokens=program_tokens, program_edge_mask=program_edge_mask\n                )\n                final_program_state = encoded[3]\n        else:\n            for encoder in self.encoders:\n                encoded = encoder(encoded)\n            final_program_state = None\n        node_features = encoded[0]     # (B, N, node_dim)\n        \n        # Program-based prediction\n        y_pred, program_info = self.program_head(\n            node_features, type_exp_center, encoder_program_state=final_program_state,\n            program_mask=program_mask,\n        )\n        \n        return y_pred, program_info\n\n\nclass SpaGP_Loss(nn.Module):\n    """\n    Loss function for SpaGP model:\n        L = L_mse + λ_orth * L_orthogonality + λ_sparse * L_sparsity\n    """\n    def __init__(self, gene_list, interaction_gene_list1, \n                 lambda_orth=0.1, lambda_sparse=0.01):\n        super().__init__()\n        interaction_gene_list = list(set(elem for sublist in interaction_gene_list1[0] for elem in sublist))\n        \n        self.interaction_gene_index = []\n        self.not_interaction_gene_index = []\n        for i in range(len(gene_list)):\n            if gene_list[i] in interaction_gene_list:\n                self.interaction_gene_index.append(i)\n            else:\n                self.not_interaction_gene_index.append(i)\n        self.interaction_gene_index = torch.LongTensor(self.interaction_gene_index)\n        self.not_interaction_gene_index = torch.LongTensor(self.not_interaction_gene_index)\n        \n        self.mse = nn.MSELoss()\n        self.lambda_orth = lambda_orth\n        self.lambda_sparse = lambda_sparse\n    \n    def orthogonality_loss(self, programs):\n        """\n        Soft orthogonality penalty: Σ_{i≠j} (p_i · p_j)²\n        \n        Args:\n            programs: (K, genes) — normalized program vectors\n        """\n        # Gram matrix: (K, K)\n        gram = programs @ programs.T  # p_i · p_j for all pairs\n        # Zero out diagonal (self-similarity = 1, we don\'t penalize that)\n        K = programs.shape[0]\n        if K < 2:\n            # A single program has no program pair, so orthogonality/diversity\n            # is defined exactly as zero rather than evaluating 0 / (K * (K-1)).\n            return programs.new_zeros(())\n        mask = 1.0 - torch.eye(K, device=programs.device)\n        off_diag = gram * mask\n        # Penalty: sum of squared off-diagonal entries\n        return torch.sum(off_diag ** 2) / (K * (K - 1))\n    \n    def sparsity_loss(self, activations):\n        """\n        L1 sparsity on program activations.\n        Encourages each cell to activate only a few programs.\n        \n        Args:\n            activations: (B, K) — non-negative program activations\n        """\n        return torch.mean(activations)  # L1 of non-negative = mean\n    \n    def forward(self, y_pred_tuple, y):\n        """\n        Args:\n            y_pred_tuple: (y_pred, program_info) from model\n            y: (B, genes) ground truth\n        \n        Returns:\n            total_loss: scalar (for backward)\n            mse_not_interact: scalar (for monitoring, detached)\n            orth_loss: scalar (for monitoring, detached)\n            sparse_loss: scalar (for monitoring, detached)\n        """\n        y_pred, program_info = y_pred_tuple\n        \n        # Main prediction loss\n        mse_loss = self.mse(y_pred, y)\n        \n        # Orthogonality penalty\n        orth_loss = self.orthogonality_loss(program_info[\'programs\'])\n        \n        # Sparsity penalty\n        sparse_loss = self.sparsity_loss(program_info[\'activations\'])\n        \n        # Total\n        total_loss = mse_loss + self.lambda_orth * orth_loss + self.lambda_sparse * sparse_loss\n        \n        # Monitoring: MSE on non-interaction genes\n        mse_not_interact = self.mse(\n            y_pred[:, self.not_interaction_gene_index],\n            y[:, self.not_interaction_gene_index]\n        ).detach().cpu()\n        \n        return (total_loss, mse_not_interact, \n                orth_loss.detach().cpu(), sparse_loss.detach().cpu())\nclass SpaGP_NoSpatial(nn.Module):\n    """\n    Ablation model: g_k predicted from center cell expression only.\n    No neighborhood information → if VE ≈ full model, spatial story fails.\n    """\n    def __init__(self, genes, ligands_info, node_dim=256, edge_dim=48, num_heads=2,\n                 n_layers=1, att_dim=8, use_cell_type_embedding=True,\n                 n_programs=16):\n        super().__init__()\n        self.genes = genes\n        self.n_programs = n_programs\n        n_genes = len(genes)\n        \n        # Programs (same as SpaGP)\n        self.programs = nn.Parameter(torch.empty(n_programs, n_genes))\n        nn.init.xavier_uniform_(self.programs)\n        \n        # Activation from center cell expression ONLY (no spatial context)\n        self.activation_net = nn.Sequential(\n            nn.Linear(n_genes, 128),\n            nn.GELU(),\n            nn.Linear(128, n_programs),\n            nn.Softplus()\n        )\n    \n    def forward(self, x):\n        # Only use center cell\'s raw expression — NO neighbors\n        center_exp = x["type_exp"][:, 0, :] + x["x"][:, 0, :]  # (B, genes)\n        \n        g = self.activation_net(center_exp)  # (B, K)\n        programs_norm = F.normalize(self.programs, dim=-1)\n        y_pred = g @ programs_norm\n        \n        program_info = {\n            \'activations\': g,\n            \'programs\': programs_norm,\n            \'context\': center_exp,\n        }\n        return y_pred, program_info\n', 'spagat/dataloader.py': 'import torch\nfrom torch.utils.data import Dataset\nimport os\nimport numpy as np\nimport pandas as pd\n\nclass SPAGAT_dataset(Dataset):\n    def __init__(self, processed_dir, num_neighbors=50, col_centerx="centerx",\n                 col_centery="centery", col_cell_type="subclass"):\n        if processed_dir[-1] != "/":\n            processed_dir = processed_dir + "/"\n\n        samples = []\n        for filei in os.listdir(processed_dir):\n            if filei.find("_TypeExp.npz") >= 0:\n                samples.append(filei.split("_TypeExp.npz")[0])\n        self.samples = list(sorted(list(set(samples))))\n        print("Have samples:", len(self.samples), self.samples)\n\n        self.index_index = ["index_" + str(i) for i in range(num_neighbors)]\n        self.interactions = torch.load("/".join(processed_dir.split("/")[:-2]) + "/ligands.pth", weights_only=False)\n        self.genes = torch.load("/".join(processed_dir.split("/")[:-2]) + "/genes.pth", weights_only=False)\n\n        # Collect all cell types from all data files\n        all_cell_types = set()\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                all_cell_types.update(dfi[col_cell_type].unique())\n            except Exception as e:\n                print(f"Warning: Cannot read sample {samplei}: {e}")\n                continue\n        # Build cell type dictionary, sorted for consistency\n        self.cell_types_dict = {ct: i for i, ct in enumerate(sorted(all_cell_types))}\n        print(f"Built cell type dictionary with {len(self.cell_types_dict)} cell types: {sorted(all_cell_types)}")\n\n        self.indexes = []\n        self.flags = []\n        self.centerx = []\n        self.centery = []\n        self.exps = []\n        self.cell_types = []\n        self.type_exp = []\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                # Assign cell type numbers for each cell in the sample\n                dfi["cell_type_number"] = np.array([self.cell_types_dict.get(cell_type, -1)\n                                                     for cell_type in dfi[col_cell_type]])\n                if "flag" not in dfi.columns.tolist():\n                    dfi["flag"] = [True for _ in range(dfi.shape[0])]\n\n                type_exp_dicti = np.load(processed_dir + samplei + "_TypeExp.npz", allow_pickle=True)\n                type_exp_dicti = {k: torch.Tensor(np.array(v).astype(np.float64)) for k, v in type_exp_dicti.items()}\n                # Only keep cell types present in both the dict and the global cell type dict\n                valid_cell_types = [ct for ct in dfi[col_cell_type].unique()\n                                   if ct in type_exp_dicti and ct in self.cell_types_dict]\n                type_exp = torch.stack([type_exp_dicti[cell_type] for cell_type in dfi[col_cell_type]\n                                       if cell_type in valid_cell_types], dim=0)\n                # Skip sample if no valid cell types\n                if len(valid_cell_types) == 0:\n                    print(f"Skip sample {samplei}: no valid cell types")\n                    continue\n                # Only keep rows with valid cell types\n                valid_mask = dfi[col_cell_type].isin(valid_cell_types)\n                dfi_valid = dfi[valid_mask].copy()\n                if len(dfi_valid) == 0:\n                    print(f"Skip sample {samplei}: no valid rows after filtering")\n                    continue\n                self.indexes.append(torch.LongTensor(dfi_valid.loc[:, self.index_index].values))\n                self.flags.append(dfi_valid.loc[:, "flag"].values)\n                self.centerx.append(torch.Tensor(dfi_valid.loc[:, col_centerx].values))\n                self.centery.append(torch.Tensor(dfi_valid.loc[:, col_centery].values))\n                self.exps.append(torch.Tensor(dfi_valid.loc[:, self.genes].values))\n                self.cell_types.append(torch.LongTensor(dfi_valid.loc[:, "cell_type_number"].values))\n                # Rebuild type_exp for valid cell types\n                type_exp_valid = torch.stack([type_exp_dicti[cell_type] for cell_type in dfi_valid[col_cell_type]], dim=0)\n                self.type_exp.append(type_exp_valid)\n            except Exception as e:\n                print(f"Error: Cannot process sample {samplei}: {e}")\n                import traceback\n                traceback.print_exc()\n                continue\n\n        self.meta_counts = []\n        self.arg_meta = []\n        for i in range(len(self.samples)):\n            if i < len(self.flags):\n                self.meta_counts.append(np.sum(self.flags[i]))\n                self.arg_meta.append(torch.LongTensor(np.where(self.flags[i] != 0)[0]))\n        self.meta_counts = np.array(self.meta_counts) if self.meta_counts else np.array([])\n        self.cumsum_meta = np.cumsum(self.meta_counts) if len(self.meta_counts) > 0 else np.array([])\n        print("There are totally", self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0, "cells in this dataset")\n\n    def __len__(self):\n        return self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0\n\n    def __getitem__(self, idx):\n        if len(self.cumsum_meta) == 0:\n            raise RuntimeError("No valid data in dataset")\n        # Find which sample this idx comes from\n        sample_id = np.searchsorted(self.cumsum_meta, idx, side=\'right\')\n        idx = idx - self.cumsum_meta[sample_id]\n        idx = self.arg_meta[sample_id][idx]\n        indices = self.indexes[sample_id][idx]\n        centerx = self.centerx[sample_id][indices]\n        centery = self.centery[sample_id][indices]\n        exp = self.exps[sample_id][indices]\n        type_exp = self.type_exp[sample_id][indices]\n        y = exp[0]\n        cell_types = torch.LongTensor(self.cell_types[sample_id][indices])\n        num_real_neighbors = len(indices)\n        neighbor_mask = torch.ones(num_real_neighbors)\n        return {\n            "x": exp,\n            "type_exp": type_exp,\n            "y": y,\n            "cell_types": cell_types,\n            "position_x": centerx,\n            "position_y": centery,\n            "neighbor_mask": neighbor_mask\n        }\n\nclass SPAGAT_evaluate_dataset(Dataset):\n    def __init__(self, processed_dir, sample, num_neighbors=50, col_centerx="centerx",\n                 col_centery="centery", col_cell_type="subclass"):\n        if processed_dir[-1] != "/":\n            processed_dir = processed_dir + "/"\n        self.samples = [sample]\n        print("Have samples:", self.samples)\n        self.index_index = ["index_" + str(i) for i in range(num_neighbors)]\n        self.interactions = torch.load("/".join(processed_dir.split("/")[:-2]) + "/ligands.pth", weights_only=False)\n        self.genes = torch.load("/".join(processed_dir.split("/")[:-2]) + "/genes.pth", weights_only=False)\n        # Collect all cell types from all data files\n        all_cell_types = set()\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                all_cell_types.update(dfi[col_cell_type].unique())\n            except Exception as e:\n                print(f"Warning: Cannot read sample {samplei}: {e}")\n                continue\n        # Build cell type dictionary, sorted for consistency\n        self.cell_types_dict = {ct: i for i, ct in enumerate(sorted(all_cell_types))}\n        print(f"Built cell type dictionary with {len(self.cell_types_dict)} cell types: {sorted(all_cell_types)}")\n        self.indexes = []\n        self.flags = []\n        self.centerx = []\n        self.centery = []\n        self.exps = []\n        self.cell_types = []\n        self.type_exp = []\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                dfi["cell_type_number"] = np.array([self.cell_types_dict.get(cell_type, -1)\n                                                     for cell_type in dfi[col_cell_type]])\n                if "flag" not in dfi.columns.tolist():\n                    dfi["flag"] = [True for _ in range(dfi.shape[0])]\n                type_exp_dicti = np.load(processed_dir + samplei + "_TypeExp.npz", allow_pickle=True)\n                type_exp_dicti = {k: torch.Tensor(np.array(v).astype(np.float64)) for k, v in type_exp_dicti.items()}\n                type_exp = torch.stack([type_exp_dicti[cell_type] for cell_type in dfi[col_cell_type]], dim=0)\n                self.type_exp.append(type_exp)\n                self.indexes.append(torch.LongTensor(dfi.loc[:, self.index_index].values))\n                self.flags.append(dfi.loc[:, "flag"].values)\n                self.centerx.append(torch.Tensor(dfi.loc[:, col_centerx].values))\n                self.centery.append(torch.Tensor(dfi.loc[:, col_centery].values))\n                self.exps.append(torch.Tensor(dfi.loc[:, self.genes].values))\n                self.cell_types.append(torch.LongTensor(dfi.loc[:, "cell_type_number"].values))\n            except Exception as e:\n                print(f"Error: Cannot process sample {samplei}: {e}")\n                import traceback\n                traceback.print_exc()\n                continue\n        self.meta_counts = []\n        self.arg_meta = []\n        for i in range(len(self.samples)):\n            if i < len(self.flags):\n                self.meta_counts.append(np.sum(self.flags[i]))\n                self.arg_meta.append(torch.LongTensor(np.where(self.flags[i] != 0)[0]))\n        self.meta_counts = np.array(self.meta_counts) if self.meta_counts else np.array([])\n        self.cumsum_meta = np.cumsum(self.meta_counts) if len(self.meta_counts) > 0 else np.array([])\n        print("There are totally", self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0, "cells in this dataset")\n\n    def __len__(self):\n        return self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0\n\n    def __getitem__(self, idx):\n        if len(self.cumsum_meta) == 0:\n            raise RuntimeError("No valid data in dataset")\n        sample_id = np.searchsorted(self.cumsum_meta, idx, side=\'right\')\n        idx = idx - self.cumsum_meta[sample_id]\n        idx = self.arg_meta[sample_id][idx]\n        indices = self.indexes[sample_id][idx]\n        centerx = self.centerx[sample_id][indices]\n        centery = self.centery[sample_id][indices]\n        exp = self.exps[sample_id][indices]\n        type_exp = self.type_exp[sample_id][indices]\n        y = exp[0]\n        cell_types = torch.LongTensor(self.cell_types[sample_id][indices])\n        indices = torch.LongTensor(indices)\n        return {\n            "x": exp,\n            "type_exp": type_exp,\n            "y": y,\n            "cell_types": cell_types,\n            "position_x": centerx,\n            "position_y": centery,\n            "NN": indices\n        }\n', 'scripts/train.py': '"""SpaGAT inductive revision using a fixed train/validation/test split.\n\nSelect checkpoints on validation median gene-wise PCC only, then evaluate the\nbest checkpoint once on test per seed. Model and metric definitions are retained.\n"""\n\nimport argparse\nimport json\nimport os\nimport random\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader, Subset\n\nimport sys\n\nPROJECT_ROOT = os.path.abspath(\n    os.path.join(os.path.dirname(__file__), "..")\n)\n\nif PROJECT_ROOT not in sys.path:\n    sys.path.insert(0, PROJECT_ROOT)\n\nfrom spagat.dataloader import SPAGAT_dataset\nfrom spagat.gene_program_model import SpaGP, SpaGP_Loss\n\n\ndef set_all_seeds(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef percentile_summary(tensor):\n    """JSON-serializable distribution summary; None denotes unavailable q."""\n    if tensor is None or tensor.numel() == 0:\n        return None\n    values = tensor.detach().float().flatten().cpu()\n    quantiles = torch.quantile(values, torch.tensor([0.05, 0.25, 0.5, 0.75, 0.95]))\n    return {\n        "mean": values.mean().item(),\n        "std": values.std(unbiased=False).item(),\n        "min": values.min().item(),\n        "p05": quantiles[0].item(),\n        "p25": quantiles[1].item(),\n        "median": quantiles[2].item(),\n        "p75": quantiles[3].item(),\n        "p95": quantiles[4].item(),\n        "max": values.max().item(),\n        "positive_fraction": (values > 0).float().mean().item(),\n        "negative_fraction": (values < 0).float().mean().item(),\n    }\n\n\ndef gene_wise_pcc(prediction, target):\n    prediction = prediction.float()\n    target = target.float()\n    prediction = prediction - prediction.mean(dim=0, keepdim=True)\n    target = target - target.mean(dim=0, keepdim=True)\n    denominator = torch.sqrt((prediction.square().sum(dim=0) * target.square().sum(dim=0)).clamp_min(1e-12))\n    pcc = (prediction * target).sum(dim=0) / denominator\n    return pcc.clamp(-1, 1)\n\n\ndef explained_variance_percent(prediction, target):\n    residual_sum = (target - prediction).square().sum()\n    total_sum = target.square().sum().clamp_min(1e-12)  # zero-residual reference; logging only\n    return (100 * (1 - residual_sum / total_sum)).item()\n\n\ndef routing_statistics(attention_batches):\n    """Compute conditional neighbor entropy and program-routing cosine similarity."""\n    if not attention_batches:\n        return None, None\n    attention = torch.cat(attention_batches, dim=0).float()  # (cells, N-1, K)\n\n    # The public tensor excludes the center self-edge.  Renormalize it so entropy\n    # describes the conditional distribution over retained neighboring cells.\n    conditional = attention / attention.sum(dim=1, keepdim=True).clamp_min(1e-12)\n    entropy = -(conditional.clamp_min(1e-12) * conditional.clamp_min(1e-12).log()).sum(dim=1)\n    entropy_per_program = entropy.mean(dim=0)\n\n    # K=1 has a well-defined entropy but no pair of programs to compare.\n    # Return N/A for routing similarity; these metrics are logging-only.\n    if conditional.shape[-1] < 2:\n        return entropy_per_program, None\n\n    routing_vectors = conditional.permute(2, 0, 1).reshape(conditional.shape[-1], -1)\n    routing_vectors = routing_vectors / routing_vectors.norm(dim=1, keepdim=True).clamp_min(1e-12)\n    similarity = routing_vectors @ routing_vectors.T\n    return entropy_per_program, similarity\n\n\ndef evaluate(model, loader, loss_function, device, return_predictions=False):\n    model.eval()\n    predictions, targets = [], []\n    activations, coefficients, attention_batches = [], [], []\n    loss_sum, steps = 0.0, 0\n\n    with torch.no_grad():\n        for batch in loader:\n            batch = {name: value.to(device) for name, value in batch.items()}\n            model_output = model(batch)\n            loss, _, _, _ = loss_function(model_output, batch["y"])\n            prediction, info = model_output\n\n            loss_sum += loss.item()\n            steps += 1\n            predictions.append(prediction.cpu())\n            targets.append(batch["y"].cpu())\n            if not return_predictions:\n                activations.append(info["activations"].cpu())\n            if not return_predictions and info.get("program_message_coefficients") is not None:\n                coefficients.append(info["program_message_coefficients"].cpu())\n            if not return_predictions and info.get("program_attention") is not None:\n                attention_batches.append(info["program_attention"].cpu())\n\n    prediction = torch.cat(predictions, dim=0)\n    target = torch.cat(targets, dim=0)\n    pcc = gene_wise_pcc(prediction, target)\n    # Final test uses the same metric functions, in one inference pass. Avoid\n    # retaining diagnostic attention tensors for the much larger held-out set.\n    if return_predictions:\n        return {\n            "test_loss": loss_sum / steps,\n            "test_mse": torch.mean((prediction - target).square()).item(),\n            "test_mse_zero": torch.mean(target.square()).item(),\n            "median_gene_pcc": torch.median(pcc).item(),\n            "mean_gene_pcc": torch.mean(pcc).item(),\n            "ev_percent": explained_variance_percent(prediction, target),\n            "n_cells": int(target.shape[0]),\n            "n_genes": int(target.shape[1]),\n        }, prediction, target, pcc\n    entropy_per_program, routing_similarity = routing_statistics(attention_batches)\n    off_diagonal_similarity = None\n    if routing_similarity is not None and routing_similarity.shape[0] > 1:\n        mask = ~torch.eye(routing_similarity.shape[0], dtype=torch.bool)\n        off_diagonal_similarity = routing_similarity[mask].mean().item()\n\n    return {\n        "val_loss": loss_sum / steps,\n        "val_mse": torch.mean((prediction - target).square()).item(),\n        "median_gene_pcc": torch.median(pcc).item(),\n        "mean_gene_pcc": torch.mean(pcc).item(),\n        "ev_percent": explained_variance_percent(prediction, target),\n        "g_distribution": percentile_summary(torch.cat(activations, dim=0)),\n        "q_distribution": percentile_summary(torch.cat(coefficients, dim=0) if coefficients else None),\n        "attention_entropy_per_program": None if entropy_per_program is None else entropy_per_program.tolist(),\n        "attention_entropy_mean": None if entropy_per_program is None else entropy_per_program.mean().item(),\n        "routing_similarity": None if routing_similarity is None else routing_similarity.tolist(),\n        "routing_similarity_off_diagonal_mean": off_diagonal_similarity,\n    }\n\n\ndef load_and_validate_shared_split(split_path, dataset_size, data_dir=None):\n    """Load a previously saved split and verify it is valid for this dataset."""\n    with open(split_path, "r", encoding="utf-8") as handle:\n        split = json.load(handle)\n    keys = ("train_indices", "val_indices", "test_indices")\n    if not isinstance(split, dict) or not set(keys).issubset(split):\n        raise ValueError(\n            f"{split_path} must contain train_indices, val_indices and test_indices."\n        )\n\n    if any(not isinstance(split[key], list) or not split[key] for key in keys):\n        raise ValueError("Each split must be a non-empty JSON list.")\n    all_indices = [index for key in keys for index in split[key]]\n    if any(type(index) is not int for index in all_indices):\n        raise ValueError("Shared split must contain non-empty integer index lists.")\n    if any(index < 0 or index >= dataset_size for index in all_indices):\n        raise ValueError(\n            f"Shared split contains an index outside the current dataset range [0, {dataset_size - 1}]."\n        )\n    if len(set(all_indices)) != len(all_indices):\n        raise ValueError("Shared split contains duplicate or overlapping indices.")\n    if len(all_indices) != dataset_size:\n        raise ValueError("Fixed split must cover the current dataset exactly once.")\n    if data_dir is not None and "processed_dir" in split:\n        if os.path.realpath(split["processed_dir"]) != os.path.realpath(data_dir):\n            raise ValueError("--data-dir does not match the split\'s processed_dir.")\n    for key, count_key in zip(keys, ("n_train", "n_validation", "n_test")):\n        if count_key in split and split[count_key] != len(split[key]):\n            raise ValueError(f"{count_key} does not match {key}.")\n    return split  # Preserve the inductive split\'s provenance metadata.\n\n\ndef validate_sample_order(split_path, dataset, split):\n    """Check the saved sample audit when present, without fetching any cells."""\n    audit_path = os.path.splitext(split_path)[0] + "_samples.csv"\n    if not os.path.isfile(audit_path):\n        return\n    audit = pd.read_csv(audit_path)\n    if audit["sample"].tolist() != list(dataset.samples):\n        raise ValueError("Dataset sample order differs from the saved split audit.")\n    counts = np.asarray(dataset.meta_counts, dtype=np.int64)\n    ends = np.cumsum(counts)\n    starts = ends - counts\n    for column, expected in (("eligible_cells", counts),\n                             ("global_start", starts),\n                             ("global_end_exclusive", ends)):\n        if not np.array_equal(audit[column].to_numpy(), expected):\n            raise ValueError(f"Dataset {column} differs from the saved split audit.")\n    membership = np.empty(len(dataset), dtype=np.int8)\n    labels = ("train", "validation", "test")\n    for label, key in enumerate(("train_indices", "val_indices", "test_indices")):\n        membership[split[key]] = label\n    for row, start, end in zip(audit.itertuples(index=False), starts, ends):\n        if row.split not in labels or not np.all(membership[start:end] == labels.index(row.split)):\n            raise ValueError(f"Sample {row.sample} differs from the fixed split.")\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description="Train the SpaGAT model for residual cell-state prediction.")\n    parser.add_argument("--data-dir", required=True)\n    parser.add_argument("--output-dir", required=True)\n    parser.add_argument("--seeds", type=int, nargs="+", default=[123])\n    parser.add_argument("--split-seed", type=int, default=123)\n    \n\n    parser.add_argument(\n        "--shared-split", required=True,\n        help="Fixed inductive JSON with train_indices, val_indices and test_indices.",\n    )\n    parser.add_argument(\n    "--max-epochs", "--epochs", dest="max_epochs", type=int, default=10,\n    help="Maximum training epochs (default: 10). --epochs is a compatibility alias.",\n)\n    parser.add_argument(\n        "--patience", type=int, default=5,\n        help="Stop after this many consecutive epochs without improving validation median gene-wise PCC.",\n    )\n    parser.add_argument("--batch-size", type=int, default=32)\n    parser.add_argument("--lr", type=float, default=1e-4)\n    parser.add_argument("--num-neighbors", type=int, default=50)\n    parser.add_argument("--node-dim", type=int, default=256)\n    parser.add_argument("--edge-dim", type=int, default=48)\n    parser.add_argument("--num-heads", type=int, default=2)\n    parser.add_argument("--n-layers", type=int, default=1)\n    parser.add_argument("--att-dim", type=int, default=8)\n    parser.add_argument("--n-programs", type=int, default=4)\n    parser.add_argument("--program-token-dim", type=int, default=32)\n    \n    parser.add_argument("--lambda-orth", type=float, default=0.1)\n    parser.add_argument("--lambda-sparse", type=float, default=0.01)\n    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n    parser.add_argument("--component-ablation", choices=["full","no_distance"], default="full")\n    parser.add_argument("--routing-mode", choices=["program"], default="program")\n    parser.add_argument("--neighbor-cache", default=None)\n    return parser.parse_args()\n\n\ndef main():\n    args = parse_args()\n    if args.max_epochs < 1:\n        raise ValueError("--max-epochs must be at least 1.")\n    if args.patience < 0:\n        raise ValueError("--patience must be non-negative.")\n    if len(set(args.seeds)) != len(args.seeds):\n        raise ValueError("Duplicate seeds would overwrite results.")\n    \n    expected = (4, "program")\n    if (args.n_programs, args.routing_mode) != expected:\n        raise ValueError("Ablation definition/configuration mismatch")\n    device = torch.device(args.device)\n    # Each invocation owns a new output directory; never reuse old results.\n    os.makedirs(args.output_dir, exist_ok=False)\n    data_dir = args.data_dir.rstrip("/")\n    data_root = os.path.dirname(data_dir)\n    genes = torch.load(os.path.join(data_root, "genes.pth"), weights_only=False)\n    ligands_info = torch.load(os.path.join(data_root, "ligands.pth"), weights_only=False)\n\n    # Construct once; deterministic index sets are reused for every selected\n    # mode/seed.  A supplied split is loaded exactly instead of regenerated.\n    if args.neighbor_cache is None:\n        dataset = SPAGAT_dataset(processed_dir=data_dir, num_neighbors=args.num_neighbors)\n    else:\n        from neighborhood_support import dataset_for_k\n        dataset = dataset_for_k(data_dir, args.num_neighbors, args.neighbor_cache)\n    split = load_and_validate_shared_split(args.shared_split, len(dataset), data_dir)\n    validate_sample_order(args.shared_split, dataset, split)\n    with open(os.path.join(args.output_dir, "shared_split.json"), "w", encoding="utf-8") as handle:\n        json.dump(split, handle)\n\n    records = []\n    best_run_records = []\n   \n    for seed in args.seeds:\n\n            set_all_seeds(seed)\n\n            model = SpaGP(\n                genes=genes,\n                ligands_info=ligands_info,\n                node_dim=args.node_dim,\n                edge_dim=args.edge_dim,\n                num_heads=args.num_heads,\n                n_layers=args.n_layers,\n                att_dim=args.att_dim,\n                n_programs=args.n_programs,\n                program_aware=True,\n                program_token_dim=args.program_token_dim,\n                message_decoder="free",\n                routing_mode=args.routing_mode,\n            ).to(device)\n            model.embeddings.component_ablation = args.component_ablation\n            optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, betas=(0.99, 0.999))\n            loss_function = SpaGP_Loss(\n                genes, ligands_info, lambda_orth=args.lambda_orth, lambda_sparse=args.lambda_sparse\n            ).to(device)\n\n            # Each mode starts with the same generator seed, giving identical\n            # shuffled batch order for every epoch under the same split.\n            loader_generator = torch.Generator().manual_seed(seed)\n            train_loader = DataLoader(\n                Subset(dataset, split["train_indices"]), batch_size=args.batch_size,\n                shuffle=True, generator=loader_generator,\n            )\n            val_loader = DataLoader(Subset(dataset, split["val_indices"]), batch_size=args.batch_size, shuffle=False)\n            parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)\n            run_start = time.perf_counter()\n            best_metric = float("-inf")\n            best_epoch = None\n            best_metrics = None\n            epochs_without_improvement = 0\n\n            for epoch in range(args.max_epochs):\n                epoch_start = time.perf_counter()\n                model.train()\n                train_loss, train_steps = 0.0, 0\n                for batch in train_loader:\n                    batch = {name: value.to(device) for name, value in batch.items()}\n                    optimizer.zero_grad(set_to_none=True)\n                    model_output = model(batch)\n                    loss, _, _, _ = loss_function(model_output, batch["y"])\n                    loss.backward()\n                    optimizer.step()\n                    train_loss += loss.item()\n                    train_steps += 1\n\n                metrics = evaluate(model, val_loader, loss_function, device)\n                monitored_metric = metrics["median_gene_pcc"]\n                improved = np.isfinite(monitored_metric) and monitored_metric > best_metric\n                if improved:\n                    best_metric = monitored_metric\n                    best_epoch = epoch\n                    epochs_without_improvement = 0\n                    best_metrics = dict(metrics)\n                else:\n                    epochs_without_improvement += 1\n                early_stopped = (\n                    not improved and epochs_without_improvement >= args.patience\n                )\n                metrics.update(\n                    {\n                        "seed": seed,\n                        "epoch": epoch,\n                        "train_loss": train_loss / train_steps,\n                        "parameter_count": parameter_count,\n                        "epoch_runtime_seconds": time.perf_counter() - epoch_start,\n                        "elapsed_runtime_seconds": time.perf_counter() - run_start,\n                        "best_epoch": best_epoch,\n                        "best_validation_pcc": best_metric,\n                        "epochs_without_improvement": epochs_without_improvement,\n                        "early_stopped": early_stopped,\n                    }\n                )\n\n                records.append(metrics)\n\n                pd.DataFrame(records).to_csv(\n                    os.path.join(args.output_dir, "training_metrics.csv"),\n                    index=False,\n                )\n\n                checkpoint = {\n                    "model": model.state_dict(),\n                    "optimizer": optimizer.state_dict(),\n                    "config": vars(args),\n                    "model_config": {\n                        "program_aware": True,\n                        "message_decoder": "free",\n                        "routing_mode": args.routing_mode,\n                        "component_ablation": args.component_ablation,\n                        "n_programs": args.n_programs,\n                        "program_token_dim": args.program_token_dim,\n                        "node_dim": args.node_dim,\n                        "edge_dim": args.edge_dim,\n                        "num_heads": args.num_heads,\n                        "n_layers": args.n_layers,\n                        "att_dim": args.att_dim,\n                    },\n                    "seed": seed,\n                    "epoch": epoch,\n                    "best_epoch": best_epoch,\n                    "best_validation_pcc": best_metric,\n                    "early_stopped": early_stopped,\n                    "epochs_without_improvement": epochs_without_improvement,\n                    "parameter_count": parameter_count,\n                    "shared_split": split,\n                    "metrics": metrics,\n                }\n\n                checkpoint_stem = f"spagat_seed{seed}"\n\n                checkpoint_path = os.path.join(\n                    args.output_dir,\n                    f"{checkpoint_stem}_latest.pth",\n                )\n\n                torch.save(\n                    checkpoint,\n                    checkpoint_path,\n                )\n\n                if improved:\n                    best_checkpoint_path = os.path.join(\n                        args.output_dir,\n                        f"{checkpoint_stem}_best.pth",\n                    )\n\n                    torch.save(\n                        checkpoint,\n                        best_checkpoint_path,\n                    )\n\n                print(\n                    f"SpaGAT seed={seed} epoch={epoch} "\n                    f"MSE={metrics[\'val_mse\']:.5f} "\n                    f"PCC={metrics[\'median_gene_pcc\']:.4f} "\n                    f"EV={metrics[\'ev_percent\']:.2f}% "\n                    f"best_epoch={best_epoch} "\n                    f"best_PCC={best_metric:.4f}"\n                )\n\n                if early_stopped:\n                    print(\n                        f"SpaGAT seed={seed} early stopping at epoch={epoch}: "\n                        f"no validation median gene-wise PCC improvement for "\n                        f"{epochs_without_improvement} epoch(s)."\n                    )\n                    break\n\n            if best_epoch is None:\n                raise RuntimeError("No finite validation PCC; test evaluation is not allowed.")\n            best_checkpoint = torch.load(best_checkpoint_path, map_location="cpu", weights_only=False)\n            model.load_state_dict(best_checkpoint["model"])\n            del best_checkpoint\n            # Test is evaluated exactly once per seed, only after selection ends.\n            test_loader = DataLoader(\n                Subset(dataset, split["test_indices"]), batch_size=args.batch_size,\n                shuffle=False, drop_last=False,\n            )\n            test_metrics, prediction, target, pcc = evaluate(\n                model, test_loader, loss_function, device, return_predictions=True\n            )\n            test_metrics.update({\n                "seed": seed, "best_epoch": best_epoch,\n                "best_validation_pcc": best_metric,\n                "checkpoint": os.path.basename(best_checkpoint_path),\n                "target_space": "residual relative to training-derived cell-type baseline",\n            })\n            test_dir = os.path.join(args.output_dir, f"seed{seed}_test")\n            os.makedirs(test_dir, exist_ok=False)\n            with open(os.path.join(test_dir, "test_metrics.json"), "x", encoding="utf-8") as handle:\n                json.dump(test_metrics, handle, indent=2)\n            pd.DataFrame([test_metrics]).to_csv(os.path.join(test_dir, "test_metrics.csv"), index=False)\n            pd.DataFrame({"gene": genes, "PCC": pcc.numpy()}).to_csv(\n                os.path.join(test_dir, "test_per_gene_PCC.csv"), index=False,\n            )\n            np.savez_compressed(\n                os.path.join(test_dir, "test_predictions.npz"),\n                prediction=prediction.numpy(), target=target.numpy(),\n                test_indices=np.asarray(split["test_indices"], dtype=np.int64),\n                genes=np.asarray(genes, dtype=str),\n            )\n            print(f"SpaGAT seed={seed} final test PCC={test_metrics[\'median_gene_pcc\']:.4f}; saved to {test_dir}")\n            del prediction, target, pcc\n\n            best_run_records.append(\n                {\n                    "seed": seed,\n                    "best_epoch": best_epoch,\n                    "best_validation_pcc": best_metric,\n                    "monitored_metric": "median_gene_pcc",\n                    "parameter_count": parameter_count,\n                    "max_epochs": args.max_epochs,\n                    "patience": args.patience,\n                }\n            )\n\n            pd.DataFrame(best_run_records).to_csv(\n                os.path.join(args.output_dir, "best_runs.csv"),\n                index=False,\n            )\nif __name__ == "__main__":\n    main()\n                \n                \n                \n                \n                \n                \n', 'neighborhood_support.py': '"""Fixed k=25/50/75 sensitivity; original expression and split stay read-only."""\nfrom pathlib import Path\nimport json, sys, hashlib, re, time\nimport numpy as np\nimport pandas as pd\nfrom scipy.spatial import cKDTree\nfrom ablation_support import sha, array_sha, save_json, metrics, load_reference, import_train\n\nKS = [25, 50, 75]\n\ndef extend_neighbors(xy, old, target=75, eligible_rows=None, audit_path=None):\n    """Preserve every original neighbor, checking these are ordered nearest cells.\n\n    Exact Euclidean neighbors for eligible receivers in this section. Stored\n    rows for excluded receivers are never fetched by the original loader and\n    do not define a graph used in training/validation/test. Their index rows\n    remain available to preserve the original row layout, with findings logged.\n    Boundary ties for new cells are resolved by row index. No expression used.\n    """\n    xy = np.asarray(xy, dtype=np.float64)\n    old = np.asarray(old, dtype=np.int64)\n    n = len(xy)\n    if xy.shape != (n, 2) or old.shape != (n, 50) or n < target + 1:\n        raise ValueError(\'Invalid coordinates/neighbor dimensions or section too small\')\n    if not np.isfinite(xy).all() or old.min() < 0 or old.max() >= n:\n        raise ValueError(\'Nonfinite coordinates or invalid original neighbor indices\')\n    if not np.array_equal(old[:, 0], np.arange(n)):\n        raise ValueError(\'Original index_0 is not the receiver; do not change this silently\')\n    if np.any(np.diff(np.sort(old, axis=1), axis=1) == 0):\n        raise ValueError(\'Duplicate original neighbors\')\n    if eligible_rows is None:\n        eligible_rows = np.arange(n)\n    eligible_rows = np.asarray(eligible_rows, dtype=np.int64)\n    if (eligible_rows.ndim != 1 or len(np.unique(eligible_rows)) != len(eligible_rows)\n            or np.any(eligible_rows < 0) or np.any(eligible_rows >= n)):\n        raise ValueError(\'Invalid eligible receiver row indices\')\n    eligible = np.zeros(n,dtype=bool);eligible[eligible_rows] = True\n    findings={\'n_section_rows\':n,\'eligible_receivers\':int(eligible.sum()),\n              \'excluded_receivers\':int((~eligible).sum()),\'tolerance_rule\':\'1e-8 * max(1, exact 50th-neighbor distance), unchanged from v1\',\n              \'eligible_order_failures\':0,\'excluded_order_failures\':0,\n              \'eligible_membership_failures\':0,\'excluded_membership_failures\':0,\n              \'order_failure_rows\':[],\'membership_failure_rows\':[],\'examples\':[],\n              \'excluded_rows_are_not_fetched_by_loader\':True,\n              \'all_original_neighbor_indices_retained\':True,\'checked_section_rows\':0,\'all_section_rows_checked\':False}\n    tree = cKDTree(xy)\n    result = np.empty((n, target), dtype=np.int64)\n    result[:, :50] = old\n    for lo in range(0, n, 2048):\n        hi = min(n, lo + 2048)\n        distances, ids = tree.query(xy[lo:hi], k=target+1, workers=1)\n        original_distances = np.linalg.norm(xy[old[lo:hi]] - xy[lo:hi, None, :], axis=2)\n        tol = 1e-8 * np.maximum(1, distances[:, 49])\n        bad_order = np.any(np.diff(original_distances, axis=1) < -tol[:, None], axis=1)\n        # A set may be correct but internally unordered; use its maximum here.\n        bad_membership = np.abs(original_distances.max(axis=1) - distances[:, 49]) > tol\n        for kind,bad in [(\'order\',bad_order),(\'membership\',bad_membership)]:\n            findings[\'eligible_\'+kind+\'_failures\']+=int(np.sum(bad & eligible[lo:hi]))\n            findings[\'excluded_\'+kind+\'_failures\']+=int(np.sum(bad & ~eligible[lo:hi]))\n            findings[kind+\'_failure_rows\'].extend((np.flatnonzero(bad)+lo).tolist())\n        for local in np.flatnonzero(bad_order | bad_membership):\n            if len(findings[\'examples\'])>=20:break\n            row=int(lo+local);raw=original_distances[local]\n            delta=np.abs(xy-xy[row]); clipped=np.sqrt(np.square(np.minimum(delta,1e4)).sum(axis=1))\n            saved_clipped=clipped[old[row]]; nearest_clipped=float(np.partition(clipped,49)[49])\n            ct=1e-8*max(1,nearest_clipped)\n            worst=int(np.argmin(np.diff(raw)))\n            findings[\'examples\'].append({\'receiver_row\':row,\'eligible_receiver\':bool(eligible[row]),\n                \'receiver_xy\':xy[row].tolist(),\'raw_order_failure\':bool(bad_order[local]),\n                \'raw_membership_failure\':bool(bad_membership[local]),\n                \'worst_adjacent_slots\':[worst,worst+1],\n                \'worst_adjacent_neighbor_rows\':old[row,[worst,worst+1]].tolist(),\n                \'worst_adjacent_distances\':raw[[worst,worst+1]].tolist(),\n                \'max_reverse_step\':max(0,float(-np.diff(raw).min())),\n                \'allowed_tolerance\':float(tol[local]),\n                \'raw_max_saved_distance\':float(raw.max()),\'raw_50th_distance\':float(distances[local,49]),\n                \'clipping_applies_to_saved_neighbors\':bool(np.any(delta[old[row]]>1e4)),\n                \'legacy_clipped_order_passes\':bool(np.all(np.diff(saved_clipped)>=-ct)),\n                \'legacy_clipped_k50_membership_passes\':bool(abs(saved_clipped.max()-nearest_clipped)<=ct)})\n        # Record diagnostics before raising, including on a stopped preparation.\n        findings[\'checked_section_rows\']=hi\n        findings[\'all_section_rows_checked\']=hi==n\n        if audit_path is not None:save_json(audit_path,findings)\n        if np.any((bad_order | bad_membership) & eligible[lo:hi]):\n            raise ValueError(\'Eligible receiver neighborhoods differ from ordered Euclidean nearest neighbors. \'\n                             \'No eligibility, coordinate, tolerance, or neighbor change was applied. \'\n                             \'See receiver_order_audit JSON in the summary ZIP.\')\n        for j, row in enumerate(range(lo, hi)):\n            candidates = ids[j]\n            # Include all cells tied at the new boundary, not an arbitrary tree subset.\n            radius = distances[j, target - 1]\n            if np.isclose(radius, distances[j, target], rtol=1e-12, atol=1e-12):\n                candidates = np.asarray(tree.query_ball_point(xy[row], radius + max(1., radius)*1e-12))\n            candidates = candidates[~np.isin(candidates, old[row])]\n            d = np.linalg.norm(xy[candidates] - xy[row], axis=1)\n            candidates = candidates[np.lexsort((candidates, d))]\n            result[row, 50:] = candidates[:target-50]\n    if audit_path is not None:save_json(audit_path,findings)\n    return result\n\ndef quantiles(values):\n    return {k: float(v) for k, v in zip([\'min\',\'p05\',\'median\',\'p95\',\'max\'],\n                                      np.quantile(values, [0,.05,.5,.95,1]))}\n\ndef verify_cache(cache, processed_dir):\n    cache = Path(cache)\n    m = json.loads((cache/\'manifest.json\').read_text())\n    if Path(m[\'processed_dir\']).resolve() != Path(processed_dir).resolve():\n        raise ValueError(\'Cache belongs to a different input directory\')\n    for path, digest in m[\'input_hashes\'].items():\n        if sha(path) != digest:\n            raise ValueError(\'Source input changed: \'+path)\n    for rec in m[\'sections\']:\n        if sha(cache/rec[\'index_file\']) != rec[\'index_sha256\']:\n            raise ValueError(\'Cached neighbor index changed: \'+rec[\'sample\'])\n    return m\n\ndef dataset_for_k(processed_dir, num_neighbors, cache_dir):\n    import torch\n    from spagat.dataloader import SPAGAT_dataset\n    if num_neighbors not in KS:\n        raise ValueError(\'The planned comparison is fixed to k=25,50,75\')\n    manifest = verify_cache(cache_dir, processed_dir)\n    data = SPAGAT_dataset(str(processed_dir), num_neighbors=50)\n    if data.samples != [r[\'sample\'] for r in manifest[\'sections\']]:\n        raise ValueError(\'Sample order changed\')\n    for i, rec in enumerate(manifest[\'sections\']):\n        a = np.load(Path(cache_dir)/rec[\'index_file\'], allow_pickle=False)\n        if not np.array_equal(a[:, :50], data.indexes[i].numpy()):\n            raise ValueError(\'Original neighbor ordering changed\')\n        if len(a) != len(data.exps[i]):\n            raise ValueError(\'Loader removed/reordered section rows\')\n        data.indexes[i] = torch.from_numpy(a[:, :num_neighbors].copy())\n    data.index_index = [\'index_\'+str(i) for i in range(num_neighbors)]\n    return data\n\ndef prepare_cache(config):\n    import torch\n    rec=config[\'reference\']; split=json.loads(Path(rec[\'split_path\']).read_text())\n    processed=Path(split[\'processed_dir\']); cache=Path(config[\'cache\']); cache.mkdir(parents=True, exist_ok=True)\n    if (cache/\'manifest.json\').exists():\n        verify_cache(cache, processed)\n        print(\'PREFLIGHT cached neighbors verified:\',rec[\'dataset\'],flush=True)\n        return\n    train=import_train(config[\'source\'])\n    data=train.SPAGAT_dataset(str(processed), num_neighbors=50)\n    train.load_and_validate_shared_split(rec[\'split_path\'],len(data),str(processed))\n    train.validate_sample_order(rec[\'split_path\'],data,split)\n    audit=pd.read_csv(rec[\'samples_path\']).set_index(\'sample\')\n    source_files=[processed.parent/\'genes.pth\',processed.parent/\'ligands.pth\']\n    source_files += [processed/(s+suffix) for s in data.samples for suffix in [\'.csv\',\'_TypeExp.npz\']]\n    before={str(p):sha(p) for p in source_files}\n    manifest={\'dataset\':rec[\'dataset\'],\'processed_dir\':str(processed), \'input_hashes\':before,\n              \'k_includes_receiver\':True,\'units\':\'stored coordinate units; physical conversion not inferred\',\n              \'coordinate_rescaling_in_this_experiment\':False,\'original_first_50_retained\':True,\n              \'same_sample_candidates_only\':True,\'sections\':[]}\n    rows=[]\n    for i,sample in enumerate(data.samples):\n        print(\'PREFLIGHT neighborhood:\',rec[\'dataset\'],sample,flush=True)\n        frame=pd.read_csv(processed/(sample+\'.csv\'),usecols=[\'centerx\',\'centery\',\'subclass\',\'flag\'])\n        if len(frame)!=len(data.exps[i]):\n            raise ValueError(\'Baseline filtering changes index interpretation: \'+sample)\n        xy=frame[[\'centerx\',\'centery\']].to_numpy(dtype=np.float64)\n        old=data.indexes[i].numpy()\n        receivers=data.arg_meta[i].numpy()\n        if not np.array_equal(receivers,np.flatnonzero(frame[\'flag\'].to_numpy()!=0)):\n            raise ValueError(\'Saved receiver flag differs from loader eligibility: \'+sample)\n        audit_path=cache/(sample+\'_receiver_order_audit.json\')\n        extended=extend_neighbors(xy,old,eligible_rows=receivers,audit_path=audit_path)\n        order_audit=json.loads(audit_path.read_text())\n        if order_audit[\'excluded_order_failures\'] or order_audit[\'excluded_membership_failures\']:\n            print(\'PREFLIGHT excluded receiver rows logged (never used as focal receivers):\',\n                  sample,order_audit[\'excluded_order_failures\'],order_audit[\'excluded_membership_failures\'],flush=True)\n        dest=cache/(sample+\'_k75.npy\'); np.save(dest,extended,allow_pickle=False)\n        manifest[\'sections\'].append({\'sample\':sample,\'rows\':len(frame),\'eligible_receivers\':int(data.meta_counts[i]),\n            \'index_file\':dest.name,\'index_sha256\':sha(dest),\'coordinate_array_sha256\':array_sha(xy),\n            \'receiver_order_audit\':audit_path.name})\n        for k in KS:\n            radii=np.linalg.norm(xy[extended[receivers,k-1]]-xy[receivers],axis=1)\n            rows.append({\'dataset\':rec[\'dataset\'],\'sample\':sample,\'partition\':audit.loc[sample,\'split\'],\n                         \'k_including_receiver\':k,\'n_receivers\':len(receivers),\n                         \'unit\':\'stored_coordinate_units\',**quantiles(radii)})\n    # Detect edits during construction. Only small index files were written.\n    if before!={str(p):sha(p) for p in source_files}:\n        raise ValueError(\'Source files changed during neighborhood preparation\')\n    pd.DataFrame(rows).to_csv(cache/\'neighborhood_radii_by_section.csv\',index=False)\n    save_json(cache/\'manifest.json\',manifest)\n    # This loader keeps all non-index tensors and receiver eligibility untouched.\n    print(\'PREFLIGHT k=25/50/75 prepared:\',rec[\'dataset\'],flush=True)\n\ndef summarize(root, references):\n    root=Path(root); rows=[]; genes_out=[]; provenance=[]\n    for rec in references:\n        split=json.loads(Path(rec[\'split_path\']).read_text())\n        full,y,genes=load_reference(rec,split)\n        base,_=metrics(full,y)\n        for k in KS:\n            path=Path(rec[\'npz\']); p=full\n            if k!=50:\n                path=root/\'runs\'/rec[\'dataset\']/f\'k{k}_seed123/seed123_test/test_predictions.npz\'\n                if not path.exists():continue\n                with np.load(path,allow_pickle=False) as z:\n                    if not np.array_equal(z[\'test_indices\'],split[\'test_indices\']):raise ValueError(\'Test receivers changed\')\n                    if list(z[\'genes\'].astype(str))!=list(genes):raise ValueError(\'Gene order changed\')\n                    if array_sha(z[\'target\'])!=rec[\'target_sha256_float32\']:raise ValueError(\'Targets changed\')\n                    p=z[\'prediction\']\n            m,corr=metrics(p,y)\n            rows.append({\'dataset\':rec[\'dataset\'],\'k_including_receiver\':k,\'seed\':123,\n                         \'n_test\':len(y),\'n_genes\':len(genes),**m,\n                         **{\'difference_vs_k50_\'+key:m[key]-base[key] for key in [\'median_gene_pcc\',\'mse\',\'ev_zero_percent\']}})\n            genes_out.extend({\'dataset\':rec[\'dataset\'],\'k_including_receiver\':k,\'seed\':123,\'gene\':str(g),\'PCC\':float(c)} for g,c in zip(genes,corr))\n            provenance.append({\'dataset\':rec[\'dataset\'],\'k\':k,\'path\':str(path),\n                               \'prediction_sha256\':array_sha(p),\'target_sha256\':array_sha(y)})\n    frame=pd.DataFrame(rows);frame.to_csv(root/\'neighborhood_test_metrics.csv\',index=False)\n    pd.DataFrame(genes_out).to_csv(root/\'neighborhood_per_gene_PCC.csv\',index=False)\n    save_json(root/\'prediction_provenance.json\',provenance)\n    success=sum((root/\'runs\'/r[\'dataset\']/f\'k{k}_seed123/SUCCESS.json\').exists() for r in references for k in [25,75])\n    save_json(root/\'summary_status.json\',{\'complete\':len(rows)==len(references)*3 and success==len(references)*2,\n         \'metric_rows\':len(rows),\'expected_rows\':len(references)*3,\'completed_new_trainings\':success,\n         \'expected_new_trainings\':len(references)*2,\'training_seed\':123,\'k_includes_receiver\':True,\n         \'scope\':\'Single-seed retrained sensitivity on fixed independent-test partitions. No across-seed SD or optimal-k claim.\',\n         \'selection\':\'Validation PCC selects the checkpoint separately within each fixed k; test does not select k.\',\n         \'physical_coordinate_units_verified\':False})\n    return frame\n\ndef coordinate_audit(root):\n    """Collect concrete metadata and bounded source excerpts; do not assign units."""\n    root=Path(root); out=root/\'coordinate_audit\';out.mkdir(exist_ok=True)\n    liver=Path(\'/content/drive/MyDrive/GITIII-main/liver_workdir/gitiii/data/processed\')\n    info={\'distance_from_input_coordinates\':\'Euclidean distance in stored centerx/centery units\',\n          \'model_stage_extra_unit_rescaling\':False,\n          \'distance_features\':[\'1/(d+1)\',\'1/(sqrt(d)+1)\',\'1/(1+d*d)\',\'exp(-d)\',\'exp(-d*d)\'],\n          \'physical_units\':\'Unresolved: raw-to-CSV conversion/source documentation must be traced.\',\n          \'notes\':\'Coordinate ranges and comments alone do not prove micrometer units.\', \'liver_metadata\':[]}\n    for sample in [\'CancerousLiver\',\'NormalLiver\']:\n        p=liver/(sample+\'.csv\')\n        if not p.exists():continue\n        f=pd.read_csv(p,usecols=[\'centerx\',\'centery\'])\n        xy=f.to_numpy(dtype=np.float64)\n        info[\'liver_metadata\'].append({\'sample\':sample,\'source\':str(p),\'n_cells\':len(f),\n               \'centerx\':quantiles(xy[:,0]),\'centery\':quantiles(xy[:,1]),\'unit\':\'unverified\'})\n    snippets=[]; candidates=[]\n    # Limited project locations, at most two directory levels; no recursive Drive scan.\n    for base in [Path(\'/content/drive/MyDrive/spagatv2\'),Path(\'/content/drive/MyDrive/GITIII-main\')]:\n        if not base.exists():continue\n        for pat in [\'*.py\',\'*.ipynb\',\'*/*.py\',\'*/*.ipynb\']:\n            candidates.extend(base.glob(pat))\n    candidates+=list(Path(\'/content/drive/MyDrive\').glob(\'*.ipynb\'))\n    pattern=re.compile(r\'centerx|centery|pixel.?size|micromet|x_centroid|y_centroid|global_px|0\\.108\',re.I)\n    for p in sorted(set(candidates))[:300]:\n        if not p.is_file() or p.stat().st_size>12_000_000:continue\n        try:\n            raw=p.read_text()\n            if p.suffix==\'.ipynb\':\n                raw=\'\\n\'.join(\'\'.join(c.get(\'source\',[])) for c in json.loads(raw).get(\'cells\',[]))\n            lines=raw.splitlines(); hits=[i for i,l in enumerate(lines) if pattern.search(l)]\n            if not hits:continue\n            selected=sorted({j for i in hits[:25] for j in range(max(0,i-2),min(len(lines),i+3))})\n            snippets.append({\'source\':str(p),\'sha256\':sha(p),\'excerpts\':[{\'line\':j+1,\'text\':lines[j][:600]} for j in selected]})\n        except (OSError,ValueError,UnicodeError) as e:\n            snippets.append({\'source\':str(p),\'read_error\':str(e)})\n    save_json(out/\'audit.json\',info);save_json(out/\'source_excerpts.json\',snippets)\n    print(\'Coordinate audit saved; physical units await source verification.\',flush=True)\n\nif __name__==\'__main__\':\n    config=json.loads(Path(sys.argv[2]).read_text())\n    if sys.argv[1]==\'prepare\':prepare_cache(config)\n    else:raise ValueError(\'Unknown operation\')\n'}
FINGERPRINT='8ac5d0a5ba8d499ebae5bd483ff9b0a3e04e4d680a1214a6f8f1978e36dfb284'
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,subprocess,sys,os,uuid,zipfile,shutil

def check_experiment(dest):
    frozen=json.loads((dest/'experiment.json').read_text())
    if frozen['fingerprint']!=FINGERPRINT:raise ValueError('Resume experiment differs from this notebook')
    if frozen['references']!=REFERENCES or frozen['protocol']!=PROTOCOL:
        raise ValueError('Notebook configuration changed after preparing this experiment')
    for rel,expected in SOURCES.items():
        if (dest/'source'/rel).read_bytes()!=expected.encode():raise ValueError('Saved source changed: '+rel)
    for r in REFERENCES:
        for field,digest in [('split_path','split_sha256'),('samples_path','samples_sha256')]:
            if hashlib.sha256(Path(r[field]).read_bytes()).hexdigest()!=r[digest]:raise ValueError('Fixed split changed: '+field)
        for field in ['checkpoint_path','npz']:
            if not Path(r[field]).is_file():raise FileNotFoundError(r[field])
    return frozen

def prepare_neighborhood_run():
    import torch
    if not torch.cuda.is_available():raise RuntimeError('请在 Colab 选择 GPU runtime。')
    if RESUME_DIR:
        dest=Path(RESUME_DIR)
    else:
        stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')
        dest=ROOT/f'neighborhood_sensitivity_{stamp}_{uuid.uuid4().hex[:6]}'
        dest.mkdir(exist_ok=False)
        for rel,source in SOURCES.items():
            p=dest/'source'/rel;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(source)
        (dest/'experiment.json').write_text(json.dumps({'fingerprint':FINGERPRINT,'protocol':PROTOCOL,
            'references':REFERENCES,'gpu':torch.cuda.get_device_name(),'original_files_modified':False},indent=2))
        with (dest/'environment.txt').open('w') as f:subprocess.run([sys.executable,'-m','pip','freeze'],stdout=f,check=True)
    check_experiment(dest)
    print('输出目录（断线后填写到 RESUME_DIR）：',dest,flush=True)
    return dest

def run_process(args,logpath):
    env=os.environ.copy();env['PYTHONUNBUFFERED']='1'
    with Path(logpath).open('w') as log:
        p=subprocess.Popen(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
        try:
            for line in p.stdout:
                log.write(line);log.flush()
                if any(t in line for t in ['PREFLIGHT','epoch=','final test','Error','Traceback','Mismatch']):print(line.rstrip(),flush=True)
            code=p.wait()
        except BaseException:
            p.terminate()
            try:p.wait(timeout=10)
            except subprocess.TimeoutExpired:p.kill();p.wait()
            raise
    if code:raise RuntimeError(f'本阶段停止；请下载摘要 ZIP 核对 {logpath}。未开始的任务不会自动执行。')

def get_support(dest):
    import importlib.util
    sys.path.insert(0,str(dest/'source'))
    spec=importlib.util.spec_from_file_location('neighborhood_support',dest/'source/neighborhood_support.py')
    support=importlib.util.module_from_spec(spec);spec.loader.exec_module(support)
    return support

def prepare_neighborhood_inputs(dest):
    check_experiment(dest)
    support=get_support(dest)
    try:
        for rec in REFERENCES:
            ds=rec['dataset'];report=dest/f'{ds}_full_preflight.json'
            if report.exists():
                old=json.loads(report.read_text())['full_runs'][0]
                if support.sha(rec['checkpoint_path'])!=old['checkpoint_sha256']:raise ValueError('Reference checkpoint changed')
            cfg={'dataset':ds,'source':str(dest/'source'),'references':[rec],'output':str(report)}
            path=dest/f'{ds}_preflight_config.json';path.write_text(json.dumps(cfg,indent=2))
            run_process([sys.executable,str(dest/'source/ablation_support.py'),str(path)],dest/f'{ds}_preflight.log')
            cfg={'source':str(dest/'source'),'reference':rec,'cache':str(dest/'neighbor_indices'/ds)}
            path=dest/f'{ds}_neighbors_config.json';path.write_text(json.dumps(cfg,indent=2))
            run_process([sys.executable,str(dest/'source/neighborhood_support.py'),'prepare',str(path)],dest/f'{ds}_neighbors.log')
        support.coordinate_audit(dest)
        support.summarize(dest,REFERENCES)
        print('输入检查完成，可运行下一格的4次训练。',flush=True)
    finally:package_neighborhood(dest)

def run_neighborhood_training(dest):
    check_experiment(dest);support=get_support(dest)
    for rec in REFERENCES:
        ds=rec['dataset'];report=json.loads((dest/f'{ds}_full_preflight.json').read_text())
        if support.sha(rec['checkpoint_path'])!=report['full_runs'][0]['checkpoint_sha256']:
            raise ValueError('Full checkpoint changed')
        split=json.loads(Path(rec['split_path']).read_text())
        support.verify_cache(dest/'neighbor_indices'/ds,split['processed_dir'])
    try:
        for rec in REFERENCES:
            ds=rec['dataset'];split=json.loads(Path(rec['split_path']).read_text())
            for k in [25,75]:
                job=dest/'runs'/ds/f'k{k}_seed123';job.parent.mkdir(parents=True,exist_ok=True)
                if (job/'SUCCESS.json').is_file():
                    print('保留已完成任务：',ds,k,flush=True);continue
                if job.exists():
                    archive=dest/'interrupted'/ds/(job.name+'_'+uuid.uuid4().hex[:8])
                    archive.parent.mkdir(parents=True,exist_ok=True);shutil.move(str(job),str(archive))
                cmd=[sys.executable,str(dest/'source/scripts/train.py'),'--data-dir',split['processed_dir'],
                     '--shared-split',rec['split_path'],'--output-dir',str(job),'--seeds','123',
                     '--max-epochs','10','--patience','5','--batch-size','32','--lr','0.0001',
                     '--num-neighbors',str(k),'--neighbor-cache',str(dest/'neighbor_indices'/ds),
                     '--node-dim','256','--edge-dim','48','--num-heads','2','--n-layers','1',
                     '--att-dim','8','--program-token-dim','32','--lambda-orth','0.1','--lambda-sparse','0.01',
                     '--n-programs','4','--routing-mode','program','--component-ablation','full','--device','cuda']
                print('开始训练：',ds,'k='+str(k),'seed=123',flush=True)
                run_process(cmd,job.parent/(job.name+'.log'))
                support.summarize(dest,REFERENCES)
                support.save_json(job/'SUCCESS.json',{'complete':True,'command':cmd,'fingerprint':FINGERPRINT})
                support.summarize(dest,REFERENCES)
                package_neighborhood(dest)
    finally:
        support.summarize(dest,REFERENCES)
        package_neighborhood(dest)
    print('四次新增训练完成。请返回 summary.zip 核对，先不修改论文。',flush=True)

def package_neighborhood(dest):
    dest=Path(dest);bundle=dest.parent/(dest.name+'_summary.zip')
    with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(dest.rglob('*')):
            if p.is_file() and p.suffix in {'.json','.csv','.py','.txt','.log'} and 'interrupted' not in p.relative_to(dest).parts:
                z.write(p,p.relative_to(dest))
    print('摘要 ZIP：',bundle,flush=True)
    return bundle


In [ ]:
run_dir=prepare_neighborhood_run()


## 4. 输入、邻域与坐标审计（这里不训练）

核对完整模型配置、全部test targets、基因顺序、保存的checkpoint与前32个预测。随后建立邻居索引缓存，检查每个切片的原50邻居。邻域半径按原坐标单位汇总；不自动改为μm。

报错时先运行最后的下载格，把ZIP返回核对；不要绕过检查或更换split。单位证据缺失会记录为未解决，不影响按原数值坐标进行已规定的k比较。


In [ ]:
prepare_neighborhood_inputs(run_dir)


## 5. 开始4次训练

执行顺序：Mouse k=25、75，然后SEA-AD k=25、75。每次完成都会保存预测、指标、日志和摘要ZIP。预检通过后再运行此格。


In [ ]:
run_neighborhood_training(run_dir)


## 6. 下载摘要（预检或训练中断后也可运行）

In [ ]:
bundle=package_neighborhood(run_dir)
from google.colab import files
files.download(str(bundle))
